# CLDT-Thread — Phase 5: Fail-Closed Safety Closure
## One Finite Bulk Action, Stale Fallback, Requalification, dan Restart/Replay

**Repository snapshot:** `7b43af2a7bfb850027638fab5e53261991685233`  
**Planning window:** 13–19 October 2026, sesuai Week 5 pada calendar repository  
**Week-6 boundary:** 20 October 2026; evidence repair only, no feature addition  
**Notebook status:** implementation workbook; tidak membuktikan bahwa command, fallback, atau hardware run sudah berhasil

Phase 5 hanya membuka satu control surface: global `bulk_rate_reduce` dengan finite TTL. Host gate boleh menyatakan actuation layak dipertimbangkan; gateway dan kedua endpoint tetap mempunyai keputusan lokal yang independen. Satu acceptance yang berhasil tidak menghapus kewajiban untuk membuktikan rejection, expiry, fallback, requalification, dan replay protection.

| Boundary | Week-5 meaning |
|---|---|
| Entry | Frozen Week-4 cross-layer result, complete identity, reconciled evidence, dan executable reproduction path tersedia |
| Physical actuation eligibility | Hanya frozen positive shadow result yang melewati predeclared acceptance; inconclusive, negative, atau non-evaluable result tetap remote-off |
| Direct result | Satu authenticated finite bulk-rate command, stale-observation safety trace, dan endpoint restart/replay rejection trace |
| Fail-closed result | Setiap invariant yang belum terbukti mempertahankan `remote_actuation_enabled == false` |
| Explicit exclusions | topology shift, ablation, SMP, power, sensor, dashboard, new node, SPI migration, multi-action control, dan pembelian baru |


## Week-5 Calendar
### Seluruh Feature Work Berakhir Sebelum 20 October

Calendar ini memakai tujuh hari yang memang dialokasikan repository untuk safety closure. Contract dan deterministic tests mengambil bagian awal; hardware hanya memperoleh sisa window setelah seluruh rejection path dapat diuji tanpa radio. Bar yang tumpang tindih menunjukkan integration hand-off pada source yang sama, bukan izin menjalankan dua physical treatment sekaligus.

![Phase 5 implementation calendar](docs/diagrams/phase5/implementation-roadmap.svg)

Mermaid source: [`docs/diagrams/phase5/implementation-roadmap.mmd`](docs/diagrams/phase5/implementation-roadmap.mmd)

| Date boundary | Work that may be open | Work that must already be closed |
|---|---|---|
| 13 October | contract binding and deterministic implementation | Week-4 freeze, model choice, service floor |
| 15 October | immutable command path integration | auth vectors, gate transitions, policy bounds, replay-store failure behavior |
| 17 October | physical stale-observation case | one normal bounded action and two endpoint acknowledgements |
| 18 October | physical endpoint restart/replay case | persist-before-apply and exact rejection fixtures |
| 19 October | reconciliation and frozen safety result | no new feature or alternate action |
| 20 October | Week 6 starts | every unproven invariant keeps actuation disabled |


## Notebook Boundary
### Exact Source Copies, Bukan Repository Adapter

Setiap code cell di bawah menyalin file repository snapshot ke `/content/cldt_scratch`. Comment TODO dan return scaffold tetap dibiarkan apa adanya. Menjalankan cell hanya membuat salinan kerja di Colab; tidak menulis kembali ke checkout, tidak mengubah manifest, dan tidak menandai pekerjaan selesai.

Paragraph setelah source menjelaskan fungsi, dependency, urutan implementasi, rejection behavior, dan evidence penutup. Nilai `e.g.` pada tabel hanya memperlihatkan bentuk jawaban. Nilai aktual berasal dari build, profile, device, dan run evidence.


In [ ]:
from pathlib import Path
Path("/content/cldt_scratch").mkdir(parents=True, exist_ok=True)


## 1. Entry Gate dari Phase 4
### Physical Control Bukan Default Kelanjutan Shadow Model

Week-5 source implementation dan deterministic safety tests tetap berguna pada semua outcome Week 4. Physical command hanya eligible ketika frozen cross-layer candidate melewati acceptance yang sudah ditetapkan sebelum held-out outcome dibuka. Network-only atau naive tidak menggantikannya setelah hasil terlihat.

| Entry evidence | Contoh format — diganti evidence aktual |
|---|---|
| Week-4 terminal interpretation | e.g. positive / inconclusive / negative / non-evaluable |
| Frozen model artifact | e.g. artifact ID + SHA-256 |
| Runtime `model_revision` semantics | e.g. state-generation counter captured per prediction |
| Primary ratio and denominator status | e.g. evaluable; one-sided LCB above 0.15 |
| Critical-service floor | e.g. proportion in `[0,1]` from frozen profile |
| Prediction interval binding | e.g. critical deadline-delivery ratio, unit proportion |
| Calibration/support rule | e.g. digest of calibration-only artifact |
| Reconciled physical runs | e.g. all admitted calibration and held-out run IDs |
| Reproduction result | e.g. exit 0 from exact archived bundle |
| Binary/profile identities | e.g. source, profile, gateway/RCP/A/B hashes |

Entry interpretation:

1. Positive and fully reconciled evidence may open physical command-path integration and the non-reportable normal-action pilot after deterministic safety gates pass.
2. Inconclusive, negative, or non-evaluable evidence leaves the physical path closed. Gate/guard/auth/replay tests may still be completed with fixed fixtures.
3. Any change to model artifact, feature allowlist, service floor, profile, command bytes, firmware, topology, or physical placement creates a new evidence block; old scores do not authorize the changed system.
4. Missing identity, leakage, or reconciliation is not a negative scientific result. It is an incomplete entry gate.

> **Cut rule:** physical `remote_actuation == true` is unavailable until every entry row resolves to archived evidence. Week 5 never uses a control run to finish Week-4 calibration.


## 2. Real Repository Owners dan Week-6 Exclusions

| Area | Direct Week-5 owners |
|---|---|
| Shared contract | `cldt_status.h`, `cldt_types.h`, `cldt_control_profile.h`, `cldt_auth.h`, `cldt_protocol.h`, and matching sources |
| Host decision | `host/fidelity_gate.*`, `host/policy.*`, `host/experiment_config.*`, `host/coordinator.*`, `host/broker_io.*`, `host/main.c` |
| Edge authority | `firmware/gateway/main/app_main.c`, `policy_guard.*`, `gateway_runtime.*`, `backhaul.*`, `thread_bridge.*`, `gateway_provisioning.*` |
| Endpoint authority | `firmware/endpoint/main/app_main.c`, `endpoint_runtime.*`, `workload.*`, `thread_transport.*` |
| Safety manifests | `stale-observation.jsonc/json`, `restart-replay.jsonc/json`, `schemas/experiment.schema.json` |
| Verification/build | host/common/tests plus firmware project/component CMake surfaces, registered tests, firmware Kconfig/sdkconfig, CI, and Week-5 subset of `reproduce.py` |
| Hardware | existing S3 gateway, dedicated C6 RCP, endpoints A/B, powered hub, four verified data cables, private AP, host, optional logic analyzer |
| Frozen inputs carried from Phase 1–4 | `cldt_clock_sync.*`, `cldt_event_trace.*`, `cldt_metrics.*`, `deadline_queue.*`, `thread_diagnostic.*`, `run_recorder.*`, `twin_model.*`, `estimator.*`, `kalman.*`, and `hardware/BOM.md` |

The carried owners are not copied again merely to make this notebook longer. Their completed tests and frozen outputs are entry dependencies; an unfinished clock, recorder, lifecycle audit, model, or diagnostic contract blocks the dependent Week-5 path and is closed in its existing owner without redesign.

Week 6 owns final repetitions, automated reproduction, limitations, and presentation. It does not own a new authentication scheme, unfinished policy action, missing replay store, untested fallback, context shift, ablation, SMP, power, or dashboard. Work unfinished on 19 October becomes an explicit limitation with remote actuation disabled.


## 3. Contract Repairs Sebelum Function Bodies

Current signatures expose real blockers. None may be hidden inside a function body or operator note.

| No. | Existing contract gap | Existing owner that must close it |
|---:|---|---|
| 1 | `model_revision` advances on accepted state changes, while the gate binds it as immutable | `cldt_types.h`, `fidelity_gate.h`, `twin_model.*` contract carried from Phase 4 |
| 2 | Gate TODO requires prediction issuance ordering and `P[2][2]`, but `cldt_fidelity_sample_t` carries neither issuance time nor covariance | `cldt_types.h`, `fidelity_gate.h` |
| 3 | `cldt_fidelity_limits_t` has no calibrated covariance limit and the reason vocabulary has no explicit covariance reason | `fidelity_gate.h` |
| 4 | One interval pair coexists with several predicted metrics; target and units must remain machine-bound | `twin_model.h`, `cldt_types.h` |
| 5 | `cldt_policy_propose()` says “no proposal,” but its API has no explicit no-proposal outcome | `policy.h` |
| 6 | Exact reduced bulk value and exact TTL are not selected by the current profile/manifest contract | `cldt_control_profile.h`, profile registry, manifest cross-field admission |
| 7 | Policy arrays are traffic-class indexed while ready manifests may contain multiple streams of one class | `experiment_config.c`, `policy.*`, `workload.*` |
| 8 | Policy payload has offsets but no single encode/decode owner | `cldt_protocol.h/.c` |
| 9 | `cldt_auth.c` source comment describes payload encryption although DESIGN/SECURITY require authenticated plaintext: 132-byte AAD and zero plaintext | `cldt_auth.h/.c`, `cldt_protocol.c` |
| 10 | Native common build does not link the pinned mbedTLS backend | `common/CMakeLists.txt`, CI dependency setup |
| 11 | Command acknowledgement/rejection payload bytes and exact status mapping are not defined | `cldt_status.h`, `cldt_types.h`, `cldt_protocol.h/.c`, runtime call path |
| 12 | Gateway runtime has a generic command queue but no bounded item contract retaining raw bytes, decoded metadata, and fan-out/ACK state | `gateway_runtime.h/.c`, `backhaul.*`, `thread_bridge.*` |
| 13 | Endpoint runtime has no authentication-context/key owner and no explicit commissioning entry point | `endpoint_runtime.h/.c`, device provisioning boundary |
| 14 | Durable replay record format, integrity/version check, commit/readback, and missing/corrupt fixture owner are not represented | `endpoint_runtime.h/.c` or one separately approved owner added before coding |
| 15 | Policy expiry and return to `safe_policy` have no periodic owner/API | endpoint supervisor and `workload.*` contract |
| 16 | Stale and replay templates cannot machine-express every threshold/case listed in their prose; `additionalProperties: false` forbids ad-hoc keys | schema plus the two manifest pairs |
| 17 | Registered tests contain no fidelity-gate, policy, gateway-guard, or replay-store target | `tests/CMakeLists.txt` and explicitly added real test owners |
| 18 | Requalification after gateway fallback has no unambiguous re-arm transition | `policy_guard.h/.c`, gateway supervisor contract |
| 19 | The host broker API can publish bytes, but command topic binding, acknowledgement/rejection correlation, and retry-versus-TTL behavior are not machine-bound by its current contract | `broker_io.h/.c`, `coordinator.*`, frozen broker/protocol contract |

A contract decision is complete only when the header/schema, source, deterministic test, recorder vocabulary, and physical evidence use the same meaning. Human labels may document the decision; unrelated existing fields are not overloaded to avoid a header or schema change.


## 4. Shared Safety Identity
### Stable Status Vocabulary
#### `common/include/cldt/cldt_status.h`


In [ ]:
%%writefile /content/cldt_scratch/cldt_status.h
#ifndef CLDT_STATUS_H
#define CLDT_STATUS_H

#ifdef __cplusplus
extern "C" {
#endif

/*
 * Status values are stable public contracts. A caller records the exact status
 * at a trust, parsing, queue, or transport boundary; it must not convert an
 * error into success merely to keep a run moving. Platform adapters may map an
 * ESP-IDF or broker error to CLDT_ERR_IO, but semantic failures below remain
 * distinct so evidence can explain why a message or policy was rejected.
 */
typedef enum {
    CLDT_OK = 0,
    /* Caller or callee pointer/range/precondition was invalid. */
    CLDT_ERR_INVALID_ARGUMENT = -1,
    /* Bounded caller-owned storage, pool, queue, or encoder output was full. */
    CLDT_ERR_NO_SPACE = -2,
    /* Bytes or state violate the current protocol or data-structure contract. */
    CLDT_ERR_MALFORMED = -3,
    /* Frame version cannot be safely interpreted by this implementation. */
    CLDT_ERR_UNSUPPORTED_VERSION = -4,
    /* Required message authenticity verification failed. */
    CLDT_ERR_AUTHENTICATION = -5,
    /* A finite deadline or policy TTL elapsed before acceptable processing. */
    CLDT_ERR_EXPIRED = -6,
    /* A logical item or policy epoch has already been accepted. */
    CLDT_ERR_DUPLICATE = -7,
    /* A record or epoch is older than the accepted monotonic sequence. */
    CLDT_ERR_OUT_OF_ORDER = -8,
    /* A syntactically valid value exceeds a declared or compiled safety bound. */
    CLDT_ERR_OUT_OF_RANGE = -9,
    /* The operation conflicts with lifecycle or ownership state. */
    CLDT_ERR_WRONG_STATE = -10,
    /* Required attachment, calibration, or evidence is not yet available. */
    CLDT_ERR_NOT_READY = -11,
    /* An external file, socket, broker, or platform operation failed. */
    CLDT_ERR_IO = -12,
    /* A well-formed observation or command is older than its freshness rule. */
    CLDT_ERR_STALE = -13,
    /* A well-formed frame belongs to a run other than the active run. */
    CLDT_ERR_WRONG_RUN = -14,
    /* Command coordinator identity differs from the commissioned authority. */
    CLDT_ERR_WRONG_AUTHORITY = -15,
    /* Intentional scaffold marker; it is never a valid experiment outcome. */
    CLDT_ERR_NOT_IMPLEMENTED = -127
} cldt_status_t;

#ifdef __cplusplus
}
#endif

#endif


Week 5 memakai status yang sudah ada sebagai kontrak evidence lintas host, gateway, dan endpoint. `CLDT_ERR_AUTHENTICATION`, `CLDT_ERR_EXPIRED`, `CLDT_ERR_DUPLICATE`, `CLDT_ERR_OUT_OF_ORDER`, `CLDT_ERR_STALE`, `CLDT_ERR_WRONG_RUN`, `CLDT_ERR_WRONG_AUTHORITY`, `CLDT_ERR_NOT_READY`, dan storage/platform mapping tidak boleh dilebur menjadi satu rejection umum. Status baru hanya layak ditambahkan bila satu kegagalan safety benar-benar tidak dapat direpresentasikan oleh enum ini; perubahan tersebut harus masuk ke encoder/decoder, fixed vectors, recorder vocabulary, dan reproduction pada commit yang sama.

| Boundary | Expected status evidence |
|---|---|
| Tag mismatch | e.g. `CLDT_ERR_AUTHENTICATION` |
| Equal accepted epoch | e.g. `CLDT_ERR_DUPLICATE` |
| Older epoch | e.g. `CLDT_ERR_OUT_OF_ORDER` |
| Other run/authority | e.g. `CLDT_ERR_WRONG_RUN` / `CLDT_ERR_WRONG_AUTHORITY` |
| TTL elapsed | e.g. `CLDT_ERR_EXPIRED` |
| Missing/corrupt replay admission | e.g. `CLDT_ERR_NOT_READY` or the frozen platform-storage mapping |


### `common/include/cldt/cldt_types.h`


In [ ]:
%%writefile /content/cldt_scratch/cldt_types.h
#ifndef CLDT_TYPES_H
#define CLDT_TYPES_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_PROTOCOL_MAGIC UINT16_C(0x434C)
#define CLDT_PROTOCOL_VERSION UINT8_C(1)
#define CLDT_WIRE_HEADER_BYTES UINT16_C(72)
#define CLDT_MAX_PAYLOAD_BYTES UINT16_C(256)
#define CLDT_AUTH_TAG_BYTES 16U
#define CLDT_TRACE_DETAIL_BYTES 24U
#define CLDT_POLICY_STREAM_COUNT 4U
#define CLDT_COMMAND_AUTHORITY_NODE_ID UINT32_C(0)

typedef uint32_t cldt_node_id_t;
typedef uint64_t cldt_run_id_t;
typedef uint32_t cldt_boot_id_t;
typedef uint32_t cldt_sequence_t;
typedef uint32_t cldt_policy_epoch_t;

typedef enum {
    CLDT_NODE_GATEWAY_HOST = 0,
    CLDT_NODE_RADIO_COPROCESSOR,
    CLDT_NODE_ROUTER_ENDPOINT,
    CLDT_NODE_LOW_POWER_ENDPOINT
} cldt_node_role_t;

typedef enum {
    CLDT_TRAFFIC_CONTROL = 0,
    CLDT_TRAFFIC_CRITICAL,
    CLDT_TRAFFIC_TELEMETRY,
    CLDT_TRAFFIC_BULK,
    CLDT_TRAFFIC_COUNT
} cldt_traffic_class_t;

typedef enum {
    CLDT_FRAME_OBSERVATION = 0,
    CLDT_FRAME_COMMAND,
    CLDT_FRAME_ACKNOWLEDGEMENT,
    CLDT_FRAME_CLOCK_SYNC,
    CLDT_FRAME_HEALTH
} cldt_frame_kind_t;

typedef enum {
    CLDT_EVENT_TASK_RELEASE = 0,
    CLDT_EVENT_TASK_START,
    CLDT_EVENT_TASK_FINISH,
    CLDT_EVENT_TASK_BLOCK,
    CLDT_EVENT_QUEUE_ENQUEUE,
    CLDT_EVENT_QUEUE_DEQUEUE,
    CLDT_EVENT_QUEUE_REJECT,
    CLDT_EVENT_POOL_EXHAUSTION,
    CLDT_EVENT_MESSAGE_SEND,
    CLDT_EVENT_MESSAGE_ACK,
    CLDT_EVENT_MESSAGE_EXPIRE,
    CLDT_EVENT_MESSAGE_COALESCE,
    CLDT_EVENT_MESSAGE_DROP,
    CLDT_EVENT_MESSAGE_DUPLICATE,
    CLDT_EVENT_LINK_CHANGE,
    CLDT_EVENT_POWER_SAMPLE,
    CLDT_EVENT_POLICY_APPLY,
    CLDT_EVENT_POLICY_REJECT,
    CLDT_EVENT_POLICY_FALLBACK,
    CLDT_EVENT_HEALTH,
    CLDT_EVENT_COUNT
} cldt_event_kind_t;

typedef enum {
    CLDT_GATE_COLD = 0,
    CLDT_GATE_OBSERVE,
    CLDT_GATE_TRUSTED,
    CLDT_GATE_ABSTAIN
} cldt_gate_state_t;

typedef enum {
    CLDT_MODEL_NAIVE = 0,
    CLDT_MODEL_NETWORK_ONLY,
    CLDT_MODEL_CROSS_LAYER,
    CLDT_MODEL_VARIANT_COUNT
} cldt_model_variant_t;

/*
 * In-memory metadata. It is not a packed wire structure. Encoding and decoding
 * must be performed field by field through cldt_protocol.h.
 *
 * Identity is frame-kind specific. For observations, acknowledgements, health,
 * and trace-bearing frames, node_id/boot_id identify the emitting device. A
 * version 1 command is one global policy datagram for every endpoint admitted
 * to the run: node_id is CLDT_COMMAND_AUTHORITY_NODE_ID and boot_id identifies
 * the host coordinator process that issued it, not a destination. The gateway
 * guards and forwards those identical bytes. Version 1 does not define
 * different authenticated command bytes per endpoint.
 */
typedef struct {
    cldt_frame_kind_t kind;
    cldt_traffic_class_t traffic_class;
    uint16_t flags;
    uint8_t hop_limit;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t transmit_local_us;
    uint64_t deadline_local_us;
} cldt_frame_meta_t;

/*
 * Decoder output borrows payload memory from the input byte buffer. The caller
 * must keep that buffer alive and unchanged while this view is in use.
 */
typedef struct {
    cldt_frame_meta_t meta;
    const uint8_t *payload;
    uint16_t payload_bytes;
    uint32_t crc32c;
    uint8_t authentication_tag[CLDT_AUTH_TAG_BYTES];
} cldt_frame_view_t;

typedef struct {
    cldt_event_kind_t kind;
    /* Every work-item event carries its class; HEALTH may use CLDT_TRAFFIC_COUNT. */
    cldt_traffic_class_t traffic_class;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t local_time_us;
    /*
     * Work-item events repeat the item's release and absolute deadline in the
     * same local monotonic clock domain as local_time_us. Non-work-item events
     * store zero in both fields. This permits stateless aggregate timing while
     * preserving raw timestamps for a separate per-item lifecycle audit.
     */
    uint64_t release_local_us;
    uint64_t deadline_local_us;
    uint32_t task_id;
    int8_t core_id;
    uint16_t queue_depth;
    int16_t link_rssi_dbm;
    uint32_t time_uncertainty_us;
    /* Fixed-size auxiliary bytes; each event kind documents its own encoding. */
    uint8_t detail[CLDT_TRACE_DETAIL_BYTES];
} cldt_trace_record_t;

typedef struct {
    uint32_t release_period_ms[CLDT_POLICY_STREAM_COUNT];
    uint32_t phase_offset_ms[CLDT_POLICY_STREAM_COUNT];
    uint16_t burst_limit[CLDT_POLICY_STREAM_COUNT];
    uint16_t batch_size[CLDT_POLICY_STREAM_COUNT];
    uint32_t token_rate_milli_pps[CLDT_POLICY_STREAM_COUNT];
    cldt_policy_epoch_t epoch;
    uint64_t issued_gateway_us;
    uint32_t ttl_ms;
} cldt_policy_t;

typedef struct {
    cldt_model_variant_t model_variant;
    uint64_t model_revision;
    uint64_t horizon_start_host_us;
    uint64_t horizon_end_host_us;
    uint64_t evaluated_host_us;
    uint64_t newest_observation_host_us;
    uint32_t sample_count;
    uint32_t model_lag_us;
    uint32_t clock_uncertainty_us;
    double relative_p95_error;
    double pdr_error_points;
    double prediction_interval_coverage;
    /* False when required horizon evidence is missing, stale, or unreconciled. */
    bool observation_integrity_valid;
    bool inside_calibrated_region;
} cldt_fidelity_sample_t;

#ifdef __cplusplus
}
#endif

#endif


The existing type surface already provides command authority node `0`, four gate states, four model variants, policy epochs, finite policy fields, and fidelity samples. Week-5 closure around this header has four parts:

1. Immutable model artifact identity remains distinct from runtime state generation. A gate binding cannot compare an artifact digest to the mutable `model_revision` counter by accident.
2. Every gate input required by source becomes representable: prediction issuance/horizon identity, `P[2][2]` or its exact calibrated uncertainty input, observation integrity, calibrated-region membership, freshness, and clock uncertainty.
3. Acknowledgement/rejection evidence receives a fixed payload contract. It carries attributable run, authority, epoch, accepted/rejected/expired/fallback disposition, exact status reason, and effective local time without exposing secrets.
4. Wire and trace enums remain bounded. A new status or event is added only when an existing value cannot truthfully represent the transition; raw reason codes never collapse to a generic success.

| Type-level check | Example result |
|---|---|
| Frozen artifact vs runtime revision | e.g. separate identities; no comparison across domains |
| Gate issuance/start/end/evaluated ordering | e.g. fixed vector PASS |
| Covariance input and calibrated limit | e.g. finite, nonnegative, same units |
| ACK/reject payload size | e.g. fixed bytes ≤ `CLDT_MAX_PAYLOAD_BYTES` |
| Unknown disposition/status | e.g. decoder rejects |


### Resolved Control Profile
#### `common/include/cldt/cldt_control_profile.h`


In [ ]:
%%writefile /content/cldt_scratch/cldt_control_profile.h
#ifndef CLDT_CONTROL_PROFILE_H
#define CLDT_CONTROL_PROFILE_H

#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * A ready manifest names one immutable control profile. The profile is resolved
 * by the host before a run starts, then its ID and digest are archived with the
 * evidence. It contains only safety-relevant selection values shared across
 * host and edge boundaries; it is not a generic configuration database.
 */
#define CLDT_CONTROL_PROFILE_ID_BYTES 65U
#define CLDT_CONTROL_PROFILE_DIGEST_BYTES 32U

typedef struct {
    /*
     * profile_id identifies the exact safety selection named by
     * treatment.control_profile in a ready manifest. calibration_id identifies
     * the separately versioned model-calibration evidence that supplies
     * residual and interval limits to the host fidelity gate.
     */
    char profile_id[CLDT_CONTROL_PROFILE_ID_BYTES];
    char calibration_id[CLDT_CONTROL_PROFILE_ID_BYTES];
    /* Version one permits actuation only from the frozen cross-layer candidate. */
    cldt_model_variant_t actuation_model_variant;

    /*
     * resolved_digest is the digest of the canonical, fully resolved profile
     * document. It prevents the same human-readable ID from silently referring
     * to different values in two evidence bundles. Digest calculation belongs
     * to the host registry/parser, not this portable validation function.
     */
    uint8_t resolved_digest[CLDT_CONTROL_PROFILE_DIGEST_BYTES];

    /* Host-side freshness and hysteresis inputs. */
    uint32_t maximum_observation_age_ms;
    uint16_t passing_windows_to_trust;

    /* Edge-side policy bounds. Compiled gateway/endpoints may be stricter. */
    uint32_t maximum_policy_ttl_ms;
    uint32_t maximum_total_rate_pps;
    uint32_t minimum_critical_period_ms;
    uint16_t maximum_bulk_burst_packets;
} cldt_control_profile_t;

/*
 * Validates only intrinsic profile shape and arithmetic safety. It performs no
 * file I/O, cryptographic digest calculation, model fitting, or device query.
 *
 * The later implementation must reject null/empty/non-terminated identifiers,
 * an all-zero digest, zero time/rate limits, and relationships that would make
 * a policy impossible to evaluate safely. It must leave caller-owned profile
 * bytes unchanged and return CLDT_ERR_NOT_IMPLEMENTED until those checks exist.
 */
cldt_status_t cldt_control_profile_validate(
    const cldt_control_profile_t *profile);

#ifdef __cplusplus
}
#endif

#endif


#### `common/src/cldt_control_profile.c`


In [ ]:
%%writefile /content/cldt_scratch/cldt_control_profile.c
#include "cldt/cldt_control_profile.h"

cldt_status_t cldt_control_profile_validate(
    const cldt_control_profile_t *profile)
{
    (void)profile;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject a null profile. Verify profile_id and calibration_id contain a
     *    non-empty NUL-terminated identifier within their fixed-size buffers;
     *    never call an unbounded string function on data loaded from a file.
     * 2. Require a valid actuation_model_variant. Version-one actuated profiles
     *    must name CLDT_MODEL_CROSS_LAYER; a failed shadow acceptance test keeps
     *    actuation disabled rather than selecting a better-looking model later.
     * 3. Require resolved_digest to contain at least one nonzero byte. Digest
     *    verification itself belongs to the host registry because this common
     *    library deliberately has no JSON, filesystem, or cryptographic backend.
     * 4. Require nonzero observation age, trust-window count, TTL, rate ceiling,
     *    critical period, and bulk burst limit. Check any derived arithmetic
     *    with overflow-safe operations before returning success.
     * 5. Keep this function deterministic and side-effect free so a profile
     *    can be validated before recorder creation, network connection, or
     *    endpoint command issuance. Add unit tests for every rejection branch.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


The profile is the immutable bridge between host calibration and edge ceilings. Source TODO validation remains the base; Week 5 additionally closes the selection that the current struct cannot yet express.

1. `profile_id`, `calibration_id`, `actuation_model_variant`, and `resolved_digest` resolve before recorder or network initialization.
2. `maximum_observation_age_ms`, `passing_windows_to_trust`, `maximum_policy_ttl_ms`, `maximum_total_rate_pps`, `minimum_critical_period_ms`, and `maximum_bulk_burst_packets` remain ceilings or gate limits, not secretly selected command values.
3. The exact one-action selection records which existing bulk field changes, its baseline value, its reduced value, and the finite TTL actually issued. Current struct has no place for that distinction; the representation is closed in this owner/registry before `policy.c`.
4. Gateway and endpoints calculate effective limits as the stricter of compiled maxima and resolved profile. A human-readable profile name never raises a compiled ceiling.
5. Refit, threshold change, action-value change, or TTL change produces a different resolved digest.

| Profile evidence | Example format |
|---|---|
| Profile and calibration identities | e.g. bounded ID strings |
| Resolved digest | e.g. 64 hexadecimal characters |
| Frozen actuation model | e.g. `CLDT_MODEL_CROSS_LAYER` |
| Exact bulk field changed | e.g. one existing `cldt_policy_t` field |
| Baseline → reduced value | e.g. measured integer pair |
| Exact issued TTL | e.g. finite milliseconds ≤ every local maximum |
| Host/edge effective limits | e.g. source of each stricter value |


## 5. Authenticated Plaintext Command
### `common/include/cldt/cldt_auth.h`


In [ ]:
%%writefile /content/cldt_scratch/cldt_auth.h
#ifndef CLDT_AUTH_H
#define CLDT_AUTH_H

#include <stddef.h>
#include <stdint.h>
#include <stdbool.h>
#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"
#include "cldt/cldt_protocol.h"

#define CLDT_AUTH_KEY_BYTES 32
#define CLDT_AUTH_NONCE_BYTES 12

typedef struct {
    uint8_t key[CLDT_AUTH_KEY_BYTES];
    bool key_loaded;
} cldt_auth_context_t;

cldt_status_t cldt_auth_init(cldt_auth_context_t *ctx);
cldt_status_t cldt_auth_load_key(cldt_auth_context_t *ctx, const uint8_t key[CLDT_AUTH_KEY_BYTES]);
void cldt_auth_build_nonce(cldt_run_id_t run_id, cldt_policy_epoch_t epoch, uint8_t nonce[CLDT_AUTH_NONCE_BYTES]);
cldt_status_t cldt_auth_sign(cldt_auth_context_t *ctx, cldt_run_id_t run_id, cldt_policy_epoch_t epoch, const uint8_t *aad, size_t aad_len, const uint8_t *payload, size_t payload_len, uint8_t tag[CLDT_AUTH_TAG_BYTES]);
cldt_status_t cldt_auth_verify(cldt_auth_context_t *ctx, cldt_run_id_t run_id, cldt_policy_epoch_t epoch, const uint8_t *aad, size_t aad_len, const uint8_t *payload, size_t payload_len, const uint8_t tag[CLDT_AUTH_TAG_BYTES]);
cldt_authenticator_t cldt_auth_as_authenticator(cldt_auth_context_t *ctx);

#endif // CLDT_AUTH_H


### `common/src/cldt/cldt_auth.c`


In [ ]:
%%writefile /content/cldt_scratch/cldt_auth.c
#include "cldt/cldt_auth.h"
#include <string.h>

#ifdef ESP_PLATFORM
#include "mbedtls/chachapoly.h"
#else
// TODO: link host-side mbedtls and include "mbedtls/chachapoly.h"
#endif

cldt_status_t cldt_auth_init(cldt_auth_context_t *ctx) {
    if (!ctx) {
        return CLDT_ERR_INVALID_ARGUMENT;
    }
    memset(ctx->key, 0, CLDT_AUTH_KEY_BYTES);
    ctx->key_loaded = false;
    
    // Key retrieval belongs to a platform provisioning adapter. This portable
    // context receives bounded key bytes only through cldt_auth_load_key().
    // TODO: sdkconfig requires: CONFIG_MBEDTLS_CHACHAPOLY_C=y, CONFIG_MBEDTLS_CHACHA20_C=y, CONFIG_MBEDTLS_POLY1305_C=y
    return CLDT_OK;
}

cldt_status_t cldt_auth_load_key(cldt_auth_context_t *ctx, const uint8_t key[CLDT_AUTH_KEY_BYTES]) {
    (void)ctx;
    (void)key;

    // TODO: implement key loading logic
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_auth_build_nonce(cldt_run_id_t run_id, cldt_policy_epoch_t epoch, uint8_t nonce[CLDT_AUTH_NONCE_BYTES]) {
    (void)run_id;
    (void)epoch;
    (void)nonce;

    // TODO: Nonce construction: run_id 8 bytes big-endian at nonce[0..7], epoch 4 bytes big-endian at nonce[8..11]
}

cldt_status_t cldt_auth_sign(
    cldt_auth_context_t *ctx,
    cldt_run_id_t run_id,
    cldt_policy_epoch_t epoch,
    const uint8_t *aad,
    size_t aad_len,
    const uint8_t *payload,
    size_t payload_len,
    uint8_t tag[CLDT_AUTH_TAG_BYTES])
{
    (void)ctx;
    (void)run_id;
    (void)epoch;
    (void)aad;
    (void)aad_len;
    (void)payload;
    (void)payload_len;
    (void)tag;

    // TODO: Sign flow: mbedtls_chachapoly_context ctx; mbedtls_chachapoly_init(&ctx); mbedtls_chachapoly_setkey(&ctx, key); mbedtls_chachapoly_encrypt_and_tag(&ctx, payload_len, nonce, aad, aad_len, payload, ciphertext_out, tag); mbedtls_chachapoly_free(&ctx)
    // TODO: Choose context lifetime/storage after measuring the pinned mbedTLS
    // version and the task/host stack budget.
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_auth_verify(
    cldt_auth_context_t *ctx,
    cldt_run_id_t run_id,
    cldt_policy_epoch_t epoch,
    const uint8_t *aad,
    size_t aad_len,
    const uint8_t *payload,
    size_t payload_len,
    const uint8_t tag[CLDT_AUTH_TAG_BYTES])
{
    (void)ctx;
    (void)run_id;
    (void)epoch;
    (void)aad;
    (void)aad_len;
    (void)payload;
    (void)payload_len;
    (void)tag;

    // TODO: Verify flow: mbedtls_chachapoly_auth_decrypt returns 0 on success.
    // Map MBEDTLS_ERR_CHACHAPOLY_AUTH_FAILED to CLDT_ERR_AUTHENTICATION and
    // preserve other failures separately.
    // TODO: RFC 8439 Section 2.8.2 test vectors: Key=808182...9e9f, Nonce=07000000...4647, Tag=1ae10b594f09e26a7e902ecbd0600691
    return CLDT_ERR_NOT_IMPLEMENTED;
}

static cldt_status_t auth_calculate_tag_wrapper(
    void *context,
    const uint8_t *bytes,
    size_t byte_count,
    uint8_t output_tag[CLDT_AUTH_TAG_BYTES])
{
    (void)context;
    (void)bytes;
    (void)byte_count;
    (void)output_tag;

    // TODO: Callback wrapper: extract run_id from wire bytes at CLDT_WIRE_RUN_ID_OFFSET (24), epoch at CLDT_WIRE_POLICY_EPOCH_OFFSET (20), both in network byte order
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_authenticator_t cldt_auth_as_authenticator(cldt_auth_context_t *ctx) {
    cldt_authenticator_t auth;
    memset(&auth, 0, sizeof(auth));
    auth.calculate_tag = auth_calculate_tag_wrapper;
    auth.context = ctx;
    return auth;
}


DESIGN and SECURITY define authenticated plaintext, not encryption of the 80-byte policy payload. The canonical authenticated bytes are header bytes 0–51 followed by policy payload bytes 72–151: 132 bytes of AAD, zero plaintext/ciphertext, and a 16-byte tag. Mbed TLS permits null input/output when the data length is zero; key, 12-byte nonce, AAD, and tag remain mandatory. References: [RFC 8439](https://www.rfc-editor.org/rfc/rfc8439), [Mbed TLS ChaChaPoly API](https://mbed-tls.readthedocs.io/projects/api/en/v3.6.3/api/file/chachapoly_8h/).

Implementation sequence:

1. `cldt_auth_load_key()` validates non-null context/key, copies exactly 32 bytes into owned storage, replaces an old key only through an explicit lifecycle, and clears temporary buffers. All-zero is not automatically a cryptographic invalid key, so key provisioning policy—not a guessed byte heuristic—decides admission.
2. `cldt_auth_build_nonce()` writes network-order `run_id` then `policy_epoch`. Zero run/epoch is rejected before signing, even though the void helper cannot report status; caller validation therefore precedes it.
3. Signing and verification use the same canonical AAD and zero-length data. The stale source comment describing ciphertext output is corrected as part of implementation, not followed literally.
4. Every Mbed TLS context is initialized, keyed, used once per command operation, and freed on every path unless a measured, synchronized owned lifetime is explicitly chosen.
5. Authentication failure maps to `CLDT_ERR_AUTHENTICATION`; setup/backend failure remains distinguishable.
6. `auth_calculate_tag_wrapper()` accepts only the canonical byte layout, extracts run/epoch with bounds and network order, and never reads wire offsets from a short buffer.
7. Retries and fan-out reuse identical authenticated bytes for one epoch. Re-signing different bytes under the same key/run/epoch is forbidden.

| Vector/result | Example format |
|---|---|
| RFC 8439 known-answer | e.g. PASS on host, S3, A, B |
| Project 132-byte AAD vector | e.g. exact hex frame/tag |
| Header/payload single-byte mutations | e.g. every case authentication failure |
| Wrong key and tag | e.g. exact failure status |
| Same nonce + changed bytes guard | e.g. construction rejected |
| Secret scan | e.g. no key material in manifest/log/evidence/Git |


### Policy Wire Ownership
#### `common/include/cldt/cldt_protocol.h`


In [ ]:
%%writefile /content/cldt_scratch/cldt_protocol.h
#ifndef CLDT_PROTOCOL_H
#define CLDT_PROTOCOL_H

#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef cldt_status_t (*cldt_authenticate_fn)(
    void *context,
    const uint8_t *bytes,
    size_t byte_count,
    uint8_t output_tag[CLDT_AUTH_TAG_BYTES]);

typedef struct {
    cldt_authenticate_fn calculate_tag;
    void *context;
} cldt_authenticator_t;

/*
 * Version 1 frame layout. Every multi-byte integer uses network byte order.
 * The two reserved bytes are transmitted as zero and rejected when nonzero.
 * These offsets are the wire contract; sizeof(cldt_frame_meta_t) is not.
 */
#define CLDT_WIRE_MAGIC_OFFSET 0U
#define CLDT_WIRE_VERSION_OFFSET 2U
#define CLDT_WIRE_KIND_OFFSET 3U
#define CLDT_WIRE_TRAFFIC_CLASS_OFFSET 4U
#define CLDT_WIRE_FLAGS_OFFSET 5U
#define CLDT_WIRE_HOP_LIMIT_OFFSET 7U
#define CLDT_WIRE_NODE_ID_OFFSET 8U
#define CLDT_WIRE_BOOT_ID_OFFSET 12U
#define CLDT_WIRE_SEQUENCE_OFFSET 16U
#define CLDT_WIRE_POLICY_EPOCH_OFFSET 20U
#define CLDT_WIRE_RUN_ID_OFFSET 24U
#define CLDT_WIRE_TRANSMIT_LOCAL_US_OFFSET 32U
#define CLDT_WIRE_DEADLINE_LOCAL_US_OFFSET 40U
#define CLDT_WIRE_PAYLOAD_BYTES_OFFSET 48U
#define CLDT_WIRE_RESERVED_OFFSET 50U
#define CLDT_WIRE_RESERVED_BYTES 2U
#define CLDT_WIRE_CRC32C_OFFSET 52U
#define CLDT_WIRE_AUTH_TAG_OFFSET 56U

/*
 * Version 1 policy payload layout. Array elements are contiguous and encoded
 * in traffic-class order from CLDT_TRAFFIC_CONTROL through CLDT_TRAFFIC_BULK.
 * The policy epoch in this payload must equal the frame metadata epoch.
 */
#define CLDT_POLICY_WIRE_RELEASE_PERIOD_OFFSET 0U
#define CLDT_POLICY_WIRE_PHASE_OFFSET 16U
#define CLDT_POLICY_WIRE_BURST_LIMIT_OFFSET 32U
#define CLDT_POLICY_WIRE_BATCH_SIZE_OFFSET 40U
#define CLDT_POLICY_WIRE_TOKEN_RATE_OFFSET 48U
#define CLDT_POLICY_WIRE_EPOCH_OFFSET 64U
#define CLDT_POLICY_WIRE_ISSUED_GATEWAY_US_OFFSET 68U
#define CLDT_POLICY_WIRE_TTL_MS_OFFSET 76U
#define CLDT_POLICY_WIRE_BYTES 80U

#if (CLDT_WIRE_AUTH_TAG_OFFSET + CLDT_AUTH_TAG_BYTES) != CLDT_WIRE_HEADER_BYTES
#error "Version 1 frame offsets do not match CLDT_WIRE_HEADER_BYTES"
#endif

#if CLDT_POLICY_STREAM_COUNT != 4U
#error "Version 1 policy layout requires exactly four traffic classes"
#endif

#if (CLDT_POLICY_WIRE_TTL_MS_OFFSET + 4U) != CLDT_POLICY_WIRE_BYTES
#error "Version 1 policy offsets do not match CLDT_POLICY_WIRE_BYTES"
#endif

/*
 * Returns the exact output size required for this payload. Zero means the
 * payload cannot be represented by the current protocol version.
 */
size_t cldt_protocol_encoded_size(size_t payload_bytes);

/*
 * Encodes one frame into caller-owned storage. No heap allocation or I/O is
 * permitted. output_bytes is written only on success.
 */
cldt_status_t cldt_protocol_encode(
    const cldt_frame_meta_t *meta,
    const uint8_t *payload,
    size_t payload_bytes,
    const cldt_authenticator_t *authenticator,
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes);

/*
 * Decodes and validates one complete datagram. The returned payload view
 * borrows memory from input. The function must reject trailing bytes,
 * truncation, an unsupported version, CRC failure, and required-auth failure.
 */
cldt_status_t cldt_protocol_decode(
    const uint8_t *input,
    size_t input_bytes,
    const cldt_authenticator_t *authenticator,
    bool authentication_required,
    cldt_frame_view_t *output_view);

/*
 * Applies freshness and ordering checks after successful decoding. Times are
 * in the gateway monotonic domain; uncertainty expands the rejection margin.
 * On an endpoint, applied_epoch must be the RAM mirror of a valid durable
 * replay record. The caller still owns issuer validation, durable advancement,
 * local limits, and atomic policy publication.
 */
cldt_status_t cldt_protocol_validate_command(
    const cldt_frame_view_t *frame,
    cldt_run_id_t active_run_id,
    cldt_policy_epoch_t applied_epoch,
    uint64_t now_gateway_us,
    uint32_t time_uncertainty_us);

#ifdef __cplusplus
}
#endif

#endif


#### `common/src/cldt/cldt_protocol.c`


In [ ]:
%%writefile /content/cldt_scratch/cldt_protocol.c
#include "cldt/cldt_protocol.h"

size_t cldt_protocol_encoded_size(size_t payload_bytes)
{
    (void)payload_bytes;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject payload_bytes above CLDT_MAX_PAYLOAD_BYTES before adding it to
     *    CLDT_WIRE_HEADER_BYTES.
     * 2. Perform the addition with an explicit overflow check even though the
     *    current bound is small; this function is the protocol's size gate.
     * 3. Return zero for every unrepresentable input. Callers must treat zero
     *    as a validation failure, never as an empty wire frame.
     * Test with 0, CLDT_MAX_PAYLOAD_BYTES, one byte above the limit, and a
     * SIZE_MAX value. No allocation belongs in this helper.
     */
    return 0U;
}

cldt_status_t cldt_protocol_encode(
    const cldt_frame_meta_t *meta,
    const uint8_t *payload,
    size_t payload_bytes,
    const cldt_authenticator_t *authenticator,
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes)
{
    (void)meta;
    (void)payload;
    (void)payload_bytes;
    (void)authenticator;
    (void)output;
    (void)output_capacity;
    (void)output_bytes;

    /*
     * IMPLEMENTATION TODO:
     * 1. Validate pointer combinations first: a zero-length payload may have a
     *    null payload pointer; a nonzero one may not. Require output_bytes.
     * 2. Ask cldt_protocol_encoded_size() for the exact size and reject a
     *    short output buffer without modifying it or output_bytes.
     * 3. Serialize each header field at the CLDT_WIRE_*_OFFSET declared in
     *    cldt_protocol.h and use network byte order for every multi-byte value.
     *    Write CLDT_WIRE_RESERVED_BYTES as zero. Do not cast output to a packed
     *    C structure: alignment, endianness, and compiler padding would make
     *    the wire contract unstable.
     * 4. Build the canonical integrity sequence by concatenating serialized
     *    header bytes 0-51 with the payload; the CRC and tag slots are omitted,
     *    not included as zero bytes. Calculate CRC-32C over that sequence and
     *    write it at CLDT_WIRE_CRC32C_OFFSET. When authentication is requested,
     *    supply the same sequence as ChaCha20-Poly1305 AAD with zero plaintext
     *    and write the resulting tag. Use bounded caller/stack storage or a
     *    documented scatter/gather helper; no heap allocation belongs here.
     * 5. Write output_bytes only after every validation and authenticator call
     *    succeeds. Add known-answer tests with fixed byte vectors.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_protocol_decode(
    const uint8_t *input,
    size_t input_bytes,
    const cldt_authenticator_t *authenticator,
    bool authentication_required,
    cldt_frame_view_t *output_view)
{
    (void)input;
    (void)input_bytes;
    (void)authenticator;
    (void)authentication_required;
    (void)output_view;

    /*
     * IMPLEMENTATION TODO:
     * 1. Check input and output pointers, then verify that input contains the
     *    fixed header before reading one field. Decode the payload length from
     *    bytes, validate its maximum, and require exact datagram length.
     * 2. Reject wrong magic, unsupported version, invalid enum values, trailing
     *    bytes, nonzero reserved bytes, and malformed flag combinations before
     *    publishing output_view. Read only through the declared offsets.
     * 3. Reconstruct the canonical integrity sequence (header bytes 0-51
     *    concatenated with payload), calculate CRC-32C, and compare it with the
     *    received value before publishing any view.
     * 4. If authentication_required is true, require an authenticator and
     *    verify the received tag over that same sequence as zero-plaintext AAD.
     * 5. Populate output_view only on success. Its payload is a borrowed view,
     *    so never copy a pointer into temporary decoder storage.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_protocol_validate_command(
    const cldt_frame_view_t *frame,
    cldt_run_id_t active_run_id,
    cldt_policy_epoch_t applied_epoch,
    uint64_t now_gateway_us,
    uint32_t time_uncertainty_us)
{
    (void)frame;
    (void)active_run_id;
    (void)applied_epoch;
    (void)now_gateway_us;
    (void)time_uncertainty_us;

    /*
     * IMPLEMENTATION TODO:
     * 1. Accept only CLDT_FRAME_COMMAND after cldt_protocol_decode() has
     *    verified integrity. Never let a health or observation frame enter the
     *    policy path merely because its payload happens to parse.
     * 2. Require exactly CLDT_POLICY_WIRE_BYTES, decode every field through the
     *    declared policy offsets, require frame.meta.run_id to equal the active
     *    run, require payload epoch to equal frame metadata epoch, and require
     *    that epoch to be strictly greater than applied_epoch. Endpoint callers
     *    must supply applied_epoch from a valid durable replay record rather
     *    than resetting it after reboot.
     * 3. Reject a zero or implausibly long TTL. Overflow-check conversion and
     *    addition before comparing issue time plus TTL to now_gateway_us after
     *    expanding the expiry margin by time_uncertainty_us.
     * 4. Return CLDT_ERR_MALFORMED for structural failure, CLDT_ERR_STALE for a
     *    well-formed command outside the accepted freshness window,
     *    CLDT_ERR_WRONG_RUN for another run identity, and CLDT_ERR_EXPIRED for
     *    elapsed TTL. Preserve DUPLICATE and OUT_OF_ORDER for epoch failures.
     * This helper does not persist state or apply a policy. The endpoint caller
     * must durably advance the accepted (run_id, epoch) before publishing the
     * new policy; failure to persist is a rejection and safe fallback. Test
     * duplicate epochs, time-wrap boundaries, and a command that arrives
     * exactly at the uncertainty-expanded expiry limit.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


The fixed 80-byte policy layout is not enough until one portable owner converts between `cldt_policy_t` and those bytes. Encoding/decoding policy arrays separately at host, gateway, and endpoint would create three interpretations.

1. One helper boundary in this existing module serializes and parses every policy field at the declared offset with network byte order and exact length.
2. Frame authentication covers the canonical 132-byte sequence. CRC is verified before decoded output is published; command authentication remains mandatory at gateway and endpoint.
3. `cldt_protocol_validate_command()` handles kind, run, epoch equality, durable highest epoch, TTL, and uncertainty-expanded time boundary. Coordinator authority node/boot validation remains a caller state check and produces `CLDT_ERR_WRONG_AUTHORITY`.
4. Decoder output is never partially published. Policy output and acknowledgement output follow the same commit-on-success rule.
5. The exactly-at-expiry rule, duplicate vs out-of-order distinction, and checked time arithmetic are fixed vectors before hardware.
6. Acknowledgement bytes are separate from command bytes; the gateway never modifies or reserializes the accepted command during fan-out.

| Fixed-vector boundary | Example expected result |
|---|---|
| 80-byte policy offsets | e.g. exact hex bytes |
| Payload epoch ≠ metadata epoch | e.g. `CLDT_ERR_MALFORMED` |
| Wrong run | e.g. `CLDT_ERR_WRONG_RUN` |
| Equal/older epoch | e.g. duplicate / out-of-order |
| TTL zero / expired / boundary | e.g. exact three statuses |
| Wrong authority boot | e.g. caller returns `CLDT_ERR_WRONG_AUTHORITY` |
| Trailing bytes or retained altered bytes | e.g. rejected |


### Common Build Boundary
#### `common/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/common_CMakeLists.txt
set(CLDT_COMMON_SOURCES
    "src/cldt_protocol.c"
    "src/cldt_clock_sync.c"
    "src/cldt_control_profile.c"
    "src/cldt_metrics.c"
    "src/cldt_event_trace.c"
    "src/cldt_auth.c"
    "src/cldt_crc32c.c"
)

if(COMMAND idf_component_register)
    idf_component_register(
        SRCS ${CLDT_COMMON_SOURCES}
        INCLUDE_DIRS "include"
    )
else()
    add_library(cldt_common STATIC ${CLDT_COMMON_SOURCES})
    target_include_directories(cldt_common
        PUBLIC
            ${CMAKE_CURRENT_SOURCE_DIR}/include
    )

    if(MSVC)
        target_compile_options(cldt_common PRIVATE /W4)
    else()
        target_compile_options(cldt_common PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
    endif()
endif()


The ESP-IDF defaults already enable ChaCha20, Poly1305, and ChaChaPoly, but the portable build currently does not include or link Mbed TLS. Completion means:

1. The pinned host dependency and exact imported target/library are discovered at configure time; missing crypto support fails configuration rather than compiling a no-op branch.
2. The ESP-IDF component declares the actual dependency required by the pinned IDF release.
3. Host and firmware compile the same project fixed vectors.
4. Compiler warnings remain enabled. No conditional branch returns `CLDT_ERR_NOT_IMPLEMENTED` on an active safety build.
5. CI records the dependency version instead of relying on whichever package happens to exist on a runner.


## 6. Host Fidelity Gate
### `host/fidelity_gate.h`


In [ ]:
%%writefile /content/cldt_scratch/fidelity_gate.h
#ifndef CLDT_HOST_FIDELITY_GATE_H
#define CLDT_HOST_FIDELITY_GATE_H

#include <stdbool.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef enum {
    CLDT_GATE_REASON_NONE = 0,
    CLDT_GATE_REASON_TOO_FEW_SAMPLES,
    CLDT_GATE_REASON_STALE_OBSERVATION,
    CLDT_GATE_REASON_MODEL_LAG,
    CLDT_GATE_REASON_RESIDUAL,
    CLDT_GATE_REASON_COVERAGE,
    CLDT_GATE_REASON_CLOCK_UNCERTAINTY,
    CLDT_GATE_REASON_OBSERVATION_INTEGRITY,
    CLDT_GATE_REASON_OUT_OF_REGION,
    CLDT_GATE_REASON_MODEL_CHANGED
} cldt_gate_reason_t;

typedef struct {
    /* All time values use host monotonic time; error limits are dimensionless. */
    uint32_t minimum_samples;
    uint32_t maximum_observation_age_ms;
    uint32_t maximum_model_lag_ms;
    uint32_t maximum_clock_uncertainty_us;
    double maximum_p95_relative_error;
    double maximum_pdr_error_points;
    double minimum_interval_coverage;
    uint16_t passing_windows_to_trust;
} cldt_fidelity_limits_t;

typedef struct {
    /* The gate owns trust state only; it never creates, serializes, or applies policy. */
    cldt_gate_state_t state;
    cldt_gate_reason_t reason;
    cldt_model_variant_t bound_model_variant;
    uint64_t bound_model_revision;
    uint16_t consecutive_passing_windows;
    uint64_t state_entered_host_us;
    uint64_t transitions;
} cldt_fidelity_gate_t;

/*
 * Initializes a fail-closed gate. A successfully initialized gate starts COLD,
 * so callers must observe and score enough physical evidence before requesting
 * actuation. Limits are immutable for a run; changing them requires invalidation.
 */
cldt_status_t cldt_fidelity_gate_init(
    cldt_fidelity_gate_t *gate,
    const cldt_fidelity_limits_t *limits,
    cldt_model_variant_t model_variant,
    uint64_t model_revision,
    uint64_t now_host_us);

/*
 * One hard failure leaves TRUSTED immediately; recovery requires hysteresis.
 * actuation_allowed is an output derived from state, never an input that can
 * force a favorable decision. Caller records state/reason alongside any policy.
 */
cldt_status_t cldt_fidelity_gate_evaluate(
    cldt_fidelity_gate_t *gate,
    const cldt_fidelity_limits_t *limits,
    const cldt_fidelity_sample_t *sample,
    uint64_t now_host_us,
    bool *actuation_allowed);

/* Forces a new evidence-collection epoch after run/model identity changes. */
void cldt_fidelity_gate_invalidate(
    cldt_fidelity_gate_t *gate,
    cldt_gate_reason_t reason,
    uint64_t now_host_us);

#ifdef __cplusplus
}
#endif

#endif


### `host/fidelity_gate.c`


In [ ]:
%%writefile /content/cldt_scratch/fidelity_gate.c
#include "fidelity_gate.h"

cldt_status_t cldt_fidelity_gate_init(
    cldt_fidelity_gate_t *gate,
    const cldt_fidelity_limits_t *limits,
    cldt_model_variant_t model_variant,
    uint64_t model_revision,
    uint64_t now_host_us)
{
    (void)gate;
    (void)limits;
    (void)model_variant;
    (void)model_revision;
    (void)now_host_us;

    /*
     * IMPLEMENTATION TODO: reject null/invalid limits, an unsupported model
     * variant, or zero revision. Clear the complete gate, bind the immutable
     * variant/revision, set COLD with TOO_FEW_SAMPLES, record the entry time,
     * and leave every counter deterministic. Return success only after all
     * limits and identity fields have been validated.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_fidelity_gate_evaluate(
    cldt_fidelity_gate_t *gate,
    const cldt_fidelity_limits_t *limits,
    const cldt_fidelity_sample_t *sample,
    uint64_t now_host_us,
    bool *actuation_allowed)
{
    (void)gate;
    (void)limits;
    (void)sample;
    (void)now_host_us;
    (void)actuation_allowed;

    // TODO: 4 states: COLD, OBSERVE, TRUSTED, ABSTAIN
    // TODO: COLD -> OBSERVE: first passing sample
    // TODO: OBSERVE -> TRUSTED: consecutive_passing_windows >= limits->passing_windows_to_trust
    // TODO: Any state -> ABSTAIN: any hard failure (immediately, first failing reason recorded)
    // TODO: ABSTAIN -> OBSERVE: first passing sample after failure. Asymmetric
    // hysteresis means one clean sample does not re-enable control.
    // TODO: OBSERVE -> OBSERVE: passing but not enough consecutive windows yet
    // TODO: TRUSTED -> TRUSTED: passing, actuation_allowed = true
    // TODO: Require sample variant/revision to equal the immutable gate binding;
    // validate issued/start/end/evaluated horizon ordering.
    // TODO: Fixed evaluation order: model/horizon identity, sample count,
    // observation integrity/age, model lag, residual, interval coverage, clock
    // uncertainty, then calibrated-region status.
    // TODO: Compare observation age with maximum_observation_age_ms using
    // checked unit conversion and subtraction.
    // TODO: Check P[2][2] (critical-PDR variance) against the calibrated limit;
    // excessive uncertainty produces ABSTAIN/MODEL_LAG.
    // TODO: Record time, state, reason, pass count, integrity/region flags, and
    // P[2][2] for every evaluation.
    
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_fidelity_gate_invalidate(
    cldt_fidelity_gate_t *gate,
    cldt_gate_reason_t reason,
    uint64_t now_host_us)
{
    if (!gate) {
        return;
    }

    (void)reason;
    (void)now_host_us;
    
    // TODO: Invalidate: reset to COLD, clear consecutive_passing_windows, timestamp transition
    // TODO: Recovery retains four states: ABSTAIN -> OBSERVE on the first clean
    // sample; only the full passing-window sequence may return to TRUSTED.
}


The gate is a deterministic four-state machine, not a score threshold hidden inside coordinator code. Header repairs from Section 3 precede function bodies.

1. `cldt_fidelity_gate_init()` validates all finite limits, binds `CLDT_MODEL_CROSS_LAYER` and the immutable artifact identity chosen by the repaired contract, clears counters, and enters `COLD` with actuation false.
2. `cldt_fidelity_gate_evaluate()` sets `actuation_allowed` false before validation. Every return path leaves it false unless the completed transition is `TRUSTED -> TRUSTED` on a passing sample.
3. Failure order is fixed: identity/horizon, sample count, integrity, age, model lag/covariance, residual, coverage, clock uncertainty, calibrated region. The first reason is reproducible.
4. Age uses checked host-monotonic subtraction and ms-to-us conversion. Future newest-observation timestamps are malformed, not fresh.
5. One hard failure from any state enters `ABSTAIN` immediately. The first clean recovery sample returns to `OBSERVE`; the contract and test fix whether that sample counts as passing window one.
6. Re-entry to `TRUSTED` requires the complete configured clean sequence. A duplicate score cannot advance hysteresis.
7. `cldt_fidelity_gate_invalidate()` is reserved for new run/model identity and returns to cold evidence collection. It never grants trust.

| Prior state | Input | Next state | Actuation |
|---|---|---|---|
| `COLD` | first complete pass | `OBSERVE` | false |
| `OBSERVE` | pass below required count | `OBSERVE` | false |
| `OBSERVE` | pass reaching required count | `TRUSTED` | defined by the frozen transition rule; tested explicitly |
| `TRUSTED` | complete pass | `TRUSTED` | true |
| any | hard failure | `ABSTAIN` | false |
| `ABSTAIN` | first clean sample | `OBSERVE` | false |

A physical command is not used to discover this table. Deterministic samples cover every state/reason, threshold just below/equal/above, time overflow, duplicate horizon, artifact mismatch, and reset.


## 7. One Explainable Host Proposal
### `host/policy.h`


In [ ]:
%%writefile /content/cldt_scratch/policy.h
#ifndef CLDT_HOST_POLICY_H
#define CLDT_HOST_POLICY_H

#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"
#include "experiment_config.h"
#include "twin_model.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    uint32_t maximum_total_rate_pps;
    uint32_t minimum_critical_period_ms;
    uint16_t maximum_bulk_burst_packets;
    uint16_t maximum_batch_size;
} cldt_policy_limits_t;

/* Produces a proposal only. Applying policy is a separate edge responsibility. */
cldt_status_t cldt_policy_propose(
    const cldt_experiment_config_t *config,
    const cldt_prediction_t *prediction,
    const cldt_policy_t *current,
    const cldt_policy_limits_t *limits,
    uint64_t issued_gateway_us,
    cldt_policy_t *proposal);

/* Checks aggregate and per-stream bounds before serialization. */
cldt_status_t cldt_policy_validate(
    const cldt_policy_t *policy,
    const cldt_policy_limits_t *limits);

#ifdef __cplusplus
}
#endif

#endif


### `host/policy.c`


In [ ]:
%%writefile /content/cldt_scratch/policy.c
#include "policy.h"

cldt_status_t cldt_policy_propose(
    const cldt_experiment_config_t *config,
    const cldt_prediction_t *prediction,
    const cldt_policy_t *current,
    const cldt_policy_limits_t *limits,
    uint64_t issued_gateway_us,
    cldt_policy_t *proposal)
{
    (void)config;
    (void)prediction;
    (void)current;
    (void)limits;
    (void)issued_gateway_us;
    (void)proposal;

    /*
     * IMPLEMENTATION TODO: begin with exactly one explainable candidate action:
     * reduce only the bulk stream rate when the prediction forecasts critical
     * service below the declared floor. Copy current policy as the baseline,
     * modify one bounded field, assign a new epoch and finite TTL, then call
     * cldt_policy_validate(). Require the prediction to carry the frozen
     * CLDT_MODEL_CROSS_LAYER variant, revision, and completed prior-horizon
     * identity. issued_gateway_us must come from a valid host-to-gateway clock
     * mapping with uncertainty inside the frozen bound; do not stamp host time
     * into a gateway-domain TTL field. Return "no proposal" when prediction is
     * absent, outside calibration, or the treatment does not permit actuation. A learned
     * controller is explicitly out of scope until this baseline is validated.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_validate(
    const cldt_policy_t *policy,
    const cldt_policy_limits_t *limits)
{
    (void)policy;
    (void)limits;

    /*
     * IMPLEMENTATION TODO: require non-null inputs, validate every period,
     * burst, batch size, token rate, epoch, and TTL, then use checked arithmetic
     * to calculate total offered rate. Preserve the critical stream's minimum
     * period and reject policy that exceeds any compiled or manifest-derived
     * ceiling. This is host-side defense in depth; the gateway repeats its own
     * checks and neither side assumes the other is trusted.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Only the bulk class changes. The exact field and value are frozen in the resolved profile/selection; the current API gap is closed before code.

1. `cldt_policy_propose()` distinguishes three outcomes without relying on an unchanged output buffer: invalid input, valid no-proposal, or one concrete proposal. The header must represent that distinction.
2. Proposal eligibility requires gated-control/safety treatment, frozen cross-layer prediction identity, a completed prior gate sample that allowed consideration, valid gateway-time mapping, and current policy identity.
3. The proposal begins as an exact copy of `current`. One predeclared bulk field changes; critical, control, telemetry, all unrelated bulk fields, and compiled ceilings remain identical.
4. Epoch advances exactly once with overflow rejection. `issued_gateway_us` comes from an uncertainty-qualified mapping. TTL is the exact predeclared finite value, not an arbitrary maximum.
5. `cldt_policy_validate()` defines traffic-class indexing and offered-rate arithmetic once. For actuated v1 manifests, the simplest non-ambiguous admission is one stream per class; a different mapping requires an explicit machine contract.
6. Host validation is defense in depth. Gateway and endpoint re-evaluate their own stricter bounds.

| Proposal diff | Example result |
|---|---|
| Changed policy fields excluding epoch/time/TTL | e.g. exactly one bulk field |
| Critical/control/telemetry bytes | e.g. identical |
| Total rate | e.g. within host, profile, gateway, endpoint ceilings |
| Epoch | e.g. current + 1, nonzero, no wrap |
| TTL | e.g. exact frozen value |
| No-proposal case | e.g. explicit successful abstention, no serialization |
| Out-of-region/stale/wrong model | e.g. no proposal |


## 8. Manifest Admission and Host Orchestration
### `host/experiment_config.h`


In [ ]:
%%writefile /content/cldt_scratch/experiment_config.h
#ifndef CLDT_HOST_EXPERIMENT_CONFIG_H
#define CLDT_HOST_EXPERIMENT_CONFIG_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

/*
 * This is the executable subset of a manifest, not a mirror of every planning
 * field in JSON. A state == "template" document is intentionally incomplete and
 * must never be converted into this structure. Parse only a completed
 * state == "ready" manifest after JSON Schema validation.
 */
#define CLDT_MAX_MANIFEST_NODES 4U
#define CLDT_MAX_WORKLOADS 4U
#define CLDT_CONFIG_DIGEST_BYTES 32U
#define CLDT_MANIFEST_ID_BYTES 65U
#define CLDT_MANIFEST_TEXT_BYTES 161U

typedef enum {
    CLDT_TREATMENT_BASELINE = 0,
    CLDT_TREATMENT_PREDICTION,
    CLDT_TREATMENT_GATED_CONTROL,
    CLDT_TREATMENT_SAFETY,
    CLDT_TREATMENT_SMP,
    CLDT_TREATMENT_POWER
} cldt_treatment_mode_t;

typedef enum {
    CLDT_ACTION_NONE = 0,
    CLDT_ACTION_BULK_RATE_REDUCE,
    /* Reserved for explicitly admitted future phases; rejected by v1 control. */
    CLDT_ACTION_PHASE_STAGGER,
    CLDT_ACTION_POWER_PROFILE
} cldt_candidate_action_t;

typedef enum {
    CLDT_SCENARIO_NONE = 0,
    CLDT_SCENARIO_LOAD_STEP,
    CLDT_SCENARIO_OBSERVATION_PAUSE,
    CLDT_SCENARIO_ENDPOINT_RESTART,
    CLDT_SCENARIO_TOPOLOGY_SHIFT
} cldt_scenario_kind_t;

/*
 * Labels remain strings until provisioning maps them to physical numeric node
 * IDs. Do not hash labels ad hoc: an implementation must reject an unknown
 * label or use one documented collision-checked mapping.
 */
typedef struct {
    char label[CLDT_MANIFEST_ID_BYTES];
    cldt_node_role_t role;
} cldt_manifest_node_t;

typedef struct {
    char id[CLDT_MANIFEST_ID_BYTES];
    char source_label[CLDT_MANIFEST_ID_BYTES];
    cldt_traffic_class_t traffic_class;
    uint32_t period_ms;
    uint16_t payload_bytes;
    uint32_t deadline_ms;
    uint16_t burst_packets;
} cldt_workload_config_t;

typedef struct {
    /*
     * run_id is assigned by the launcher only after parsing and cross-field
     * validation. It is never taken from a template. Before assignment, the
     * launcher reserves a nonzero cryptographically generated value in the
     * durable global run ledger and binds a non-secret command-key identity only
     * for an actuated run. The parser leaves it zero; the coordinator rejects zero.
     */
    char experiment_id[CLDT_MANIFEST_ID_BYTES];
    cldt_run_id_t run_id;
    /* Nonzero identity of the launcher process; assigned beside run_id. */
    cldt_boot_id_t command_authority_boot_id;
    uint32_t seed;

    cldt_manifest_node_t nodes[CLDT_MAX_MANIFEST_NODES];
    size_t node_count;
    uint8_t thread_channel;
    char placement[CLDT_MANIFEST_TEXT_BYTES];
    char firmware_reference[CLDT_MANIFEST_TEXT_BYTES];

    uint32_t warmup_s;
    uint32_t measurement_s;
    uint32_t cooldown_s;
    uint16_t repetitions;

    cldt_workload_config_t workloads[CLDT_MAX_WORKLOADS];
    size_t workload_count;

    cldt_scenario_kind_t scenario;
    uint32_t scenario_at_s;
    uint32_t scenario_duration_s;
    char scenario_target[CLDT_MANIFEST_TEXT_BYTES];

    cldt_treatment_mode_t treatment_mode;
    cldt_candidate_action_t candidate_action;
    /*
     * Identifier selected by treatment.control_profile. Parsing preserves this
     * bounded name only; the host registry must resolve it to a
     * cldt_control_profile_t, verify its digest, and record both identities in
     * the evidence bundle before coordinator initialization.
     */
    char control_profile[CLDT_MANIFEST_TEXT_BYTES];
    bool host_model_enabled;
    bool remote_actuation_enabled;

    bool counter_reconciliation_required;
    double minimum_critical_on_time_pdr;
    char negative_case[CLDT_MANIFEST_TEXT_BYTES];
    uint8_t canonical_digest[CLDT_CONFIG_DIGEST_BYTES];
} cldt_experiment_config_t;

/*
 * Parses exactly one UTF-8 ready manifest from caller-owned bytes.
 *
 * Implementation sequence:
 * - validate JSON syntax and schema first;
 * - reject state == "template" before allocating or opening I/O;
 * - copy bounded fields, preserving a precise error path;
 * - calculate canonical_digest after full validation and leave run_id plus
 *   command_authority_boot_id zero for the launcher's separate assignment step.
 *
 * The parser must reject unknown runtime fields, duplicate object keys, secrets,
 * and any null value that a ready manifest is required to replace.
 */
cldt_status_t cldt_experiment_config_parse(
    const char *json,
    size_t json_bytes,
    cldt_experiment_config_t *output);

/*
 * Performs deterministic cross-field validation after parsing.
 *
 * It verifies node/stream references, rate and deadline feasibility, scenario
 * timing, treatment permissions, and compiled safety ceilings. It must not make
 * network calls, create a directory, or mutate output state.
 */
cldt_status_t cldt_experiment_config_validate(
    const cldt_experiment_config_t *config);

#ifdef __cplusplus
}
#endif

#endif


### `host/experiment_config.c`


In [ ]:
%%writefile /content/cldt_scratch/experiment_config.c
#include "experiment_config.h"

cldt_status_t cldt_experiment_config_parse(
    const char *json,
    size_t json_bytes,
    cldt_experiment_config_t *output)
{
    (void)json;
    (void)json_bytes;
    (void)output;

    /*
     * IMPLEMENTATION TODO:
     * 1. Use a maintained JSON parser with a bounded input limit; parse exactly
     *    one UTF-8 document and reject duplicate keys rather than accepting a
     *    library-specific last-key-wins behavior.
     * 2. Validate against schemas/experiment.schema.json before conversion.
     *    This runtime parser accepts only state == "ready"; a template is a
     *    planning artifact and must never start a physical run.
     * 3. Copy strings into fixed, NUL-terminated fields only after checking the
     *    destination capacity. Require a non-empty control_profile for a ready
     *    run and preserve JSON Pointer-like error paths for the operator instead
     *    of returning a generic parse failure.
     * 4. Compute the canonical manifest digest from the original validated bytes
     *    using one documented canonicalization rule; do not include credentials.
     *    Leave output.run_id and output.command_authority_boot_id zero. The
     *    launcher assigns them only after a separate durable global-ledger
     *    reservation; parsing must not silently create nonce state or require a
     *    command key for a shadow-only run.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_experiment_config_validate(
    const cldt_experiment_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate cross-field relationships that schema shape
     * checks cannot prove: every stream source must name an endpoint, deadlines
     * must be compatible with their period, aggregate offered load must fit the
     * compiled safety ceiling, scenario time must lie inside measurement time,
     * and remote actuation must be disabled for non-control treatments. Version
     * one accepts only NONE or BULK_RATE_REDUCE and rejects the reserved phase
     * and power actions even though planning templates can name them. Reject a
     * configuration before any adapter or run directory is opened. Keep this
     * function deterministic so the same manifest has the same outcome on host
     * and in future gateway subset validation.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


Week-5 cross-field admission is stricter than schema shape:

1. `stale-observation` ready mode resolves to `CLDT_SCENARIO_OBSERVATION_PAUSE`, gated control, `CLDT_ACTION_BULK_RATE_REDUCE`, host model on, and remote actuation on only after positive Week-4 evidence.
2. `restart-replay` resolves to `CLDT_SCENARIO_ENDPOINT_RESTART`, safety mode, the same single action, and the exact target endpoint label.
3. Both manifests use four unique node IDs/roles, sources that resolve to endpoints, one unambiguous stream per actuated traffic class, scenario time wholly inside measurement, finite recovery time, and strings representable by runtime buffers.
4. Profile ID/digest, model artifact, exact action, safety-case list, key identity, and command authority evidence resolve before `run_id` assignment.
5. Ready schema documents are necessary but insufficient. Any safety detail absent from the machine contract blocks promotion instead of moving into an unversioned verbal assumption.


### `host/coordinator.h`


In [ ]:
%%writefile /content/cldt_scratch/coordinator.h
#ifndef CLDT_HOST_COORDINATOR_H
#define CLDT_HOST_COORDINATOR_H

#include <stdbool.h>
#include <stdint.h>

#include "broker_io.h"
#include "cldt/cldt_control_profile.h"
#include "estimator.h"
#include "experiment_config.h"
#include "fidelity_gate.h"
#include "policy.h"
#include "run_recorder.h"
#include "twin_model.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    cldt_experiment_config_t config;
    /*
     * Resolved profile copied only after its ID matches config.control_profile
     * and its immutable digest has been recorded by the run recorder.
     */
    cldt_control_profile_t control_profile;
    cldt_broker_io_t broker;
    cldt_run_recorder_t recorder;
    /* One frozen instance and estimator per declared comparison variant. */
    cldt_twin_model_t models[CLDT_MODEL_VARIANT_COUNT];
    cldt_estimator_t estimators[CLDT_MODEL_VARIANT_COUNT];
    cldt_fidelity_gate_t gate;
    cldt_policy_t active_policy;
    uint64_t started_host_us;
    bool stop_requested;
} cldt_coordinator_t;

cldt_status_t cldt_coordinator_init(
    cldt_coordinator_t *coordinator,
    const cldt_experiment_config_t *config,
    const cldt_control_profile_t *control_profile);

cldt_status_t cldt_coordinator_run(cldt_coordinator_t *coordinator);

void cldt_coordinator_request_stop(cldt_coordinator_t *coordinator);

#ifdef __cplusplus
}
#endif

#endif


### `host/coordinator.c`


In [ ]:
%%writefile /content/cldt_scratch/coordinator.c
#include "coordinator.h"

cldt_status_t cldt_coordinator_init(
    cldt_coordinator_t *coordinator,
    const cldt_experiment_config_t *config,
    const cldt_control_profile_t *control_profile)
{
    (void)coordinator;
    (void)config;
    (void)control_profile;

    /*
     * IMPLEMENTATION TODO:
     * 1. Require a ready, cross-field-validated config with a nonzero run ID
     *    already reserved in the durable global run ledger and a nonzero command
     *    authority boot ID, plus a valid resolved control profile. Compare the
     *    profile ID with config.control_profile
     *    using bounded strings; reject mismatch before opening a recorder or
     *    broker. The caller is responsible for checking the profile digest
     *    against the canonical registry document before this function is called.
     * 2. Copy config and profile only after validation. Record the profile ID,
     *    calibration ID, actuation-model variant, and digest beside the frozen
     *    manifest. Version one may name only the cross-layer variant for
     *    actuation; failed shadow acceptance means no actuation, not model swap.
     *    Initialize the fidelity gate and edge proposal limits from that
     *    immutable selection.
     * 3. Initialize recorder, all three model/estimator pairs, fidelity gate,
     *    policy baseline, and broker in that order. Each successful step needs a
     *    paired rollback action so a later failure leaves no partial run marked
     *    valid.
     * 4. Do not open a network adapter before the immutable run directory,
     *    manifest digest, and control-profile identity exist.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_coordinator_run(cldt_coordinator_t *coordinator)
{
    (void)coordinator;

    /*
     * IMPLEMENTATION TODO: implement one bounded event loop with this strict
     * sequence for each accepted observation: record raw bytes first; validate
     * run/digest identity; update each model only from its allowed features;
     * score all variants on identical completed prior horizons; evaluate the
     * fidelity gate only for the frozen actuation variant; and only then consider
     * a new bounded proposal. Sleep or poll with a deadline so policy expiry,
     * phase transitions, and stop requests are never starved by broker traffic.
     * Finalize through the recorder with complete, invalid, or interrupted status.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_coordinator_request_stop(cldt_coordinator_t *coordinator)
{
    (void)coordinator;

    /*
     * IMPLEMENTATION TODO: make this function only set an atomic or signal-safe
     * stop flag. The main loop owns broker close, final counter requests, metric
     * reconciliation, and recorder finalization because those operations may
     * allocate, block, or fail. A signal handler must never write evidence or
     * publish a fallback command directly.
     */
}


The Week-5 coordinator sequence is single-owner and raw-first:

1. Run directory, manifest bytes/digest, profile/calibration/model identity, ledger reservation, coordinator boot ID, and non-secret key identity exist before broker connection.
2. Three shadow models continue to score identical horizons. Only the frozen cross-layer sample enters the gate.
3. Accepted inbound bytes are recorded before decode. Completed prior horizon is reconciled and scored before gate evaluation.
4. Gate false produces no proposal. Gate true only permits `cldt_policy_propose()` to consider the one action.
5. Proposal struct becomes one fixed policy payload, one global command frame, and one authenticated immutable byte array. That array is recorded before publish.
6. Gateway acceptance/rejection and both endpoint acknowledgements are correlated by run, authority boot, epoch, and exact command digest. A partial fan-out is not global success.
7. Pending commands, expiry timers, ACK deadline, stale transition, stop, and finalization have bounded owners. Broker traffic cannot starve them.
8. A stale gate transition stops new proposals. Existing endpoint commands expire by TTL; local gateway fallback is recorded according to the chosen injection/health path.
9. Requalification uses fresh completed horizons and full hysteresis. Old pending proposal bytes are never revived.
10. Finalization preserves incomplete ACK/fallback state and marks the run invalid rather than inventing closure.


### Host Broker Command Boundary
#### `host/broker_io.h`


In [ ]:
%%writefile /content/cldt_scratch/broker_io.h
#ifndef CLDT_HOST_BROKER_IO_H
#define CLDT_HOST_BROKER_IO_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_broker_message_fn)(
    void *context,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint64_t received_host_us,
    bool retained);

typedef struct {
    void *native_client;
    cldt_broker_message_fn on_message;
    void *callback_context;
    bool connected;
} cldt_broker_io_t;

cldt_status_t cldt_broker_io_open(
    cldt_broker_io_t *io,
    const char *host,
    uint16_t port,
    cldt_broker_message_fn callback,
    void *callback_context);

cldt_status_t cldt_broker_io_poll(cldt_broker_io_t *io, uint32_t timeout_ms);

cldt_status_t cldt_broker_io_publish(
    cldt_broker_io_t *io,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint8_t qos,
    bool retained);

void cldt_broker_io_close(cldt_broker_io_t *io);

#ifdef __cplusplus
}
#endif

#endif


#### `host/broker_io.c`


In [ ]:
%%writefile /content/cldt_scratch/broker_io.c
#include "broker_io.h"

cldt_status_t cldt_broker_io_open(
    cldt_broker_io_t *io,
    const char *host,
    uint16_t port,
    cldt_broker_message_fn callback,
    void *callback_context)
{
    (void)io;
    (void)host;
    (void)port;
    (void)callback;
    (void)callback_context;

    /*
     * IMPLEMENTATION TODO: validate host, port, callback, and context; create a
     * libmosquitto (or equally maintained) client with explicit protocol version,
     * TLS configuration when used, and a stable reconnect state machine. Subscribe
     * only to the private experiment namespace. The callback must enqueue or
     * return quickly; it must not update the model or write files directly.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_broker_io_poll(cldt_broker_io_t *io, uint32_t timeout_ms)
{
    (void)io;
    (void)timeout_ms;

    /*
     * IMPLEMENTATION TODO: validate io and cap timeout_ms to a small documented
     * value so coordinator deadlines are observed even during a quiet broker.
     * Translate broker reconnect, protocol, and backpressure conditions into
     * explicit CLDT statuses. Retained command delivery is a safety event, not a
     * convenient reconnect feature. Poll must never spin indefinitely.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_broker_io_publish(
    cldt_broker_io_t *io,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint8_t qos,
    bool retained)
{
    (void)io;
    (void)topic;
    (void)payload;
    (void)payload_bytes;
    (void)qos;
    (void)retained;

    /*
     * IMPLEMENTATION TODO: reject null/oversized input and out-of-range QoS;
     * require QoS 1 for durable observations as selected by the final design;
     * reject retained payloads on command topics; and expose publish queue or
     * reconnect pressure to the coordinator. The broker adapter must not invent
     * retries that extend a policy past its TTL.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

void cldt_broker_io_close(cldt_broker_io_t *io)
{
    (void)io;

    /*
     * IMPLEMENTATION TODO: unsubscribe, disconnect with a bounded deadline,
     * destroy only the native client owned by io, clear callbacks and connected
     * state, and leave raw run files to the recorder. Closing the broker must be
     * safe after a partial open and must not make a disconnected run look valid.
     */
}


Phase 3 menutup observation receive dan raw-first callback; Week 5 memakai API yang sama untuk satu command publish dan acknowledgement/rejection receive tanpa membuat broker adapter kedua.

1. `cldt_broker_io_publish()` menerima hanya complete immutable command bytes dari coordinator, exact frozen topic/run binding, bounded size, non-retained publish, dan QoS yang sudah ditetapkan oleh broker contract. Retry milik library tidak boleh menghasilkan bytes baru atau memperpanjang TTL.
2. `cldt_broker_io_poll()` tetap bounded agar command ACK deadline, stale transition, policy expiry, stop, dan finalization tidak starve ketika broker sunyi atau reconnect.
3. Callback menyalin inbound gateway/endpoint decision bytes ke bounded owner dan kembali cepat. Raw recorder tetap menerima delivery sebelum decode/join; callback tidak mengubah gate atau active policy.
4. Retained command, wrong namespace/run, duplicate QoS delivery, reconnect, queue pressure, dan disconnect mempunyai exact status/evidence. Duplicate raw delivery boleh ada; command epoch tidak boleh apply dua kali.
5. Topic names, payload envelopes, maximum size, QoS, and ACK deadline come from the frozen Phase-3 broker contract. Notebook tidak membuat topic atau retry policy baru untuk menutup kekosongan source.

| Broker closure | Example evidence |
|---|---|
| Published command bytes | e.g. digest equals recorder and gateway receive digest |
| Retained flag | e.g. false on publish; retained inbound rejected |
| Duplicate delivery | e.g. raw row retained, no second apply |
| Poll bound | e.g. command/expiry timers remain within frozen limit |
| Disconnect/reconnect | e.g. explicit event; no automatic re-arm |


### Launcher and Ledger Boundary
#### `host/main.c`


In [ ]:
%%writefile /content/cldt_scratch/host_main.c
#include <stdio.h>
#include <stdlib.h>

#include "coordinator.h"

int main(int argc, char **argv)
{
    (void)argc;
    (void)argv;

    fprintf(stderr,
            "CLDT host scaffold: implement manifest loading, recording, "
            "modeling, and fidelity-gated coordination before use.\n");

    /*
     * IMPLEMENTATION TODO:
     * 1. Accept exactly one manifest path and one optional results-root path;
     *    print a short usage error for every other argument shape.
     * 2. Read with a bounded size, validate the JSON schema, and refuse a
     *    state == "template" manifest before any network connection is made.
     * 3. Resolve the manifest's named control profile from a versioned local
     *    registry, verify its canonical digest, and reject absent or mismatched
     *    profile/calibration identities before opening a broker connection.
     * 4. Reserve a nonzero CSPRNG run ID in the durable global run ledger,
     *    collision-check it, assign it to the validated config, and generate a
     *    nonzero coordinator-process boot ID. If remote actuation is requested,
     *    also bind the non-secret command-key identity;
     *    if ledger continuity is unavailable, require key rotation before
     *    actuation. Never guess or reuse an ID. Shadow-only runs need no key.
     * 5. Create the immutable run directory, install signal handling that only
     *    requests a stop, initialize the coordinator with both config and
     *    resolved profile, and enter its event loop.
     * 6. Return a nonzero status for invalid, interrupted, or failed runs. A
     *    successful process exit is not evidence that a result is valid.
     */
    return EXIT_FAILURE;
}


The launcher owns uniqueness and key identity, not secret provisioning:

1. The strict ready manifest and profile resolve before the durable global ledger changes.
2. A CSPRNG nonzero `run_id` is collision-checked and reserved; a nonzero coordinator boot ID is generated once for the process.
3. Actuated runs bind the non-secret identity of the active command key. Secret bytes arrive through an ignored source and never enter manifest, command audit, screenshot, or Git.
4. Loss of ledger continuity blocks actuation and requires key rotation. A host/coordinator restart creates a new run; it never resumes an old nonce space.
5. Signal handling only requests stop. Terminal evidence and fallback remain normal event-loop work.


### Week-5 Reproduction Subset
#### `host/analysis/reproduce.py`


In [ ]:
%%writefile /content/cldt_scratch/reproduce.py
import sys
import json
from pathlib import Path

def main():
    if len(sys.argv) != 2:
        print("Usage: python reproduce.py <results_dir>")
        sys.exit(1)
        
    # TODO: load manifest JSON from results_dir / "manifest.json"
    # TODO: verify manifest has state="ready" and all _todo items resolved
    # TODO: compute SHA-256 digest of manifest and compare against results_dir / "manifest.sha256"
    # TODO: load events.ndjson one JSON object per line; reject malformed, blank,
    # duplicate, or trailing non-JSON records
    # TODO: group events by (run_id, node_id, boot_id, sequence) for per-item lifecycle audit
    # TODO: for each lifecycle group, verify exactly one release event and one terminal event (ack/expire/drop)
    # TODO: count duplicate_releases, duplicate_terminals, terminal_without_release, unresolved_items
    # TODO: fit naive moving-average baseline on calibration data only
    # TODO: fit network-only model: features = [delivery_outcome, link_rssi, traffic_load]
    # TODO: fit cross-layer model from the frozen network, MAC, queue, and RTOS
    # feature allowlist
    # TODO: read manifest-defined calibration and held-out blocks; never invent a percentage split after seeing results
    # TODO: score all three models on identical held-out horizons: relative P95 error on deadline delivery ratio
    # TODO: compute primary uncertainty from run-level summaries or a whole-run cluster bootstrap
    # TODO: use within-run block bootstrap only for paired time-series uncertainty,
    # never as independent physical replication
    # TODO: compute prediction interval coverage: fraction of observations within predicted +/- 2 sigma
    # TODO: build a calibration-only support envelope and retain inside/outside status for every scored horizon
    # TODO: retain observation-integrity status; missing/stale/unreconciled horizons must not disappear silently
    # TODO: perform feature-group ablation only after the primary three-model comparison is frozen
    # TODO: generate gate characterization: state/reason vs time, trust fraction,
    # false trust, abstention/requalification latency, and P[2][2]
    # TODO: output the frozen primary metric table as CSV
    # TODO: exit nonzero if reconciliation fails (any lifecycle inconsistency)
    # TODO: use numpy for statistics, matplotlib for plots, scipy.stats for bootstrap

    print("ERROR: reproduction pipeline is a scaffold and produced no result.", file=sys.stderr)
    raise SystemExit(2)

if __name__ == "__main__":
    main()


Phase 5 completes the existing gate-characterization TODO without disturbing the frozen Phase-4 primary result.

1. Preflight still verifies manifest digest, raw NDJSON, item lifecycle, aggregate reconciliation, prediction/outcome identity, and source completeness.
2. Gate replay consumes the same ordered samples and frozen limits as the live host. It emits state, reason, transition time, pass count, integrity, region, age, clock uncertainty, and covariance input for every eligible horizon.
3. Command audit joins proposal, authenticated bytes, gateway decision, both endpoint decisions, persistence commit, effective apply boundary, expiry/fallback, and terminal status.
4. Stale metrics include trigger time, gate abstention latency, local fallback/expiry latency with clock qualification, observation restoration, requalification latency, and trusted-horizon fraction.
5. Replay metrics include every attempted case and zero invalid applications. Missing/corrupt durable-state fixtures remain explicit.
6. A mismatch between live and replayed gate/command outcome exits nonzero. Negative safety evidence is still output; incomplete evidence is not converted to a result.


### Host Build Surface
#### `host/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/host_CMakeLists.txt
add_executable(cldt_host
    main.c
    coordinator.c
    experiment_config.c
    twin_model.c
    estimator.c
    fidelity_gate.c
    policy.c
    broker_io.c
    run_recorder.c
    kalman.c
)

target_include_directories(cldt_host PRIVATE ${CMAKE_CURRENT_SOURCE_DIR})
target_link_libraries(cldt_host PRIVATE cldt_common)

if(MSVC)
    target_compile_options(cldt_host PRIVATE /W4)
else()
    target_compile_options(cldt_host PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
endif()


Current host sources compile into one executable, while gate/policy tests need a linkable owner. The smallest reliable build change separates reusable host logic from the CLI or otherwise gives tests the exact same objects—never pasted duplicate implementations. JSON, MQTT, crypto, and statistical dependencies are pinned and recorded. A configuration that silently omits the authenticated command path cannot produce an actuated binary.


## 9. Deterministic Verification Surface
### `tests/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/tests_CMakeLists.txt
function(cldt_add_skeletal_test name source)
    add_executable(${name} ${source})
    target_link_libraries(${name} PRIVATE cldt_common)
    add_test(NAME ${name} COMMAND ${name})
    set_tests_properties(${name} PROPERTIES SKIP_RETURN_CODE 77)
endfunction()

cldt_add_skeletal_test(test_protocol test_protocol.c)
cldt_add_skeletal_test(test_crc32c test_crc32c.c)
cldt_add_skeletal_test(test_auth test_auth.c)
cldt_add_skeletal_test(test_clock_sync test_clock_sync.c)
cldt_add_skeletal_test(test_metrics test_metrics.c)
cldt_add_skeletal_test(test_event_trace test_event_trace.c)
cldt_add_skeletal_test(test_control_profile test_control_profile.c)


### Existing Authentication Test


In [ ]:
%%writefile /content/cldt_scratch/test_auth.c
#include <stdio.h>

#include "cldt/cldt_auth.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Use RFC 8439 known-answer material plus one project-specific fixed
     *    command vector with authority node ID 0, a fixed coordinator boot ID,
     *    the normative AAD byte order, and zero plaintext.
     * 2. Mutate every authenticated region, tag byte, run ID, epoch, and nonce
     *    byte independently and assert exact authentication failure status.
     *    A validly tagged but non-commissioned coordinator boot ID must produce
     *    CLDT_ERR_WRONG_AUTHORITY at the state-validation boundary.
     * 3. Verify one immutable command per epoch, strict epoch advance, and that
     *    retransmission reuses identical authenticated bytes rather than
     *    generating a different command under the same nonce.
     * 4. In the endpoint integration suite, verify persist-before-apply, reboot
     *    from a valid highest-epoch record, and safe fallback for missing,
     *    corrupt, or unwritable replay state. Boot identity must not substitute
     *    for that state.
     * 5. Run the same vector through the host and mbedTLS-backed targets before
     *    provisioning a command key or enabling remote actuation.
     */
    fprintf(stderr, "SKIP: authentication tests have not been implemented.\n");
    return 77;
}


### Existing Control-Profile Test


In [ ]:
%%writefile /content/cldt_scratch/test_control_profile.c
#include <stdio.h>

#include "cldt/cldt_control_profile.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Start with one completely specified in-memory profile whose IDs are
     *    bounded, digest is nonzero, and host/edge limits are all finite.
     * 2. Test one invalid condition at a time: null pointer, empty ID, missing
     *    NUL terminator, invalid/non-v1 actuation model, all-zero digest, zero
     *    freshness window, zero TTL, zero rate ceiling, zero critical period,
     *    and zero bulk burst ceiling.
     * 3. Assert exact status codes and assert the validator has not changed the
     *    input bytes. The test must not open a profile file or contact a device.
     * 4. Add a host-level test later for a manifest/profile-ID mismatch; that
     *    belongs above this portable common-library test.
     */
    fprintf(stderr, "SKIP: control profile tests have not been implemented.\n");
    return 77;
}


### Existing Protocol Test


In [ ]:
%%writefile /content/cldt_scratch/test_protocol.c
#include <stdio.h>

#include "cldt/cldt_protocol.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Begin with fixed hexadecimal byte vectors for the smallest and largest
     *    legal frames. Assert every CLDT_WIRE_*_OFFSET, zero reserved bytes,
     *    network byte order, exact size, CRC-32C, and authentication
     *    tag—not only encode/decode round trips, which can hide matching mistakes
     *    on both sides. Mutating either reserved byte must fail decoding.
     * 2. Add rejection cases one mutation at a time: wrong magic, unsupported
     *    version, header/payload length mismatch, truncation at every boundary,
     *    trailing bytes, CRC mutation, authentication mutation, and oversize data.
     * 3. Build one fixed CLDT_POLICY_WIRE_BYTES vector and assert every policy
     *    array/field offset plus equality between payload and metadata epochs.
     *    Test wrong run ID, duplicate epoch, older epoch, stale issue time, zero
     *    TTL, expired TTL, and boundary uncertainty. Verify the exact status,
     *    including CLDT_ERR_STALE and CLDT_ERR_WRONG_RUN, and confirm decoder
     *    output is not partially published.
     *    Gateway integration also rejects a nonzero command-authority node ID or
     *    wrong commissioned authority boot ID without re-encoding the datagram.
     * 4. Keep test vectors in ordinary source data with a short derivation note.
     *    Do not connect Thread or MQTT until these host-only checks are green.
     */
    fprintf(stderr, "SKIP: protocol tests have not been implemented.\n");
    return 77;
}


The three existing tests are direct Week-5 dependencies and must stop returning skip code 77. The registered surface has no gate, host policy, gateway guard, or durable replay-store test owner; real test translation units and CMake registrations are therefore added deliberately and named in the repository when created. Notebook labels do not pretend those absent files already exist.

Minimum deterministic matrix:

| Owner | Required cases before hardware |
|---|---|
| Auth | RFC/project vectors, every authenticated region mutation, wrong key/tag, nonce tuple invariant, host/S3/A/B parity |
| Protocol | policy encode/decode fixed bytes, authority/run/epoch/TTL/status boundaries, no partial output |
| Control profile | every null/zero/ID/digest/model/bound failure, input unchanged |
| Fidelity gate | all state/reason transitions, first-failure order, hysteresis, age/overflow, covariance, artifact mismatch |
| Host policy | exact one-field diff, explicit no-proposal, ceilings, epoch wrap, gateway-time uncertainty |
| Gateway guard | run/authority/epoch/TTL/health/clock/rate rejection, no mutation on failure, fallback, requalification |
| Replay store | load valid record, commit/readback, reboot, missing/corrupt/unwritable record, persist-before-apply |
| Immutable fan-out | same command bytes/digest to A and B; duplicate transport does not duplicate apply |

Every active Week-5 target exits 0 only after assertions execute. `SKIP`, `Not Run`, `CLDT_ERR_NOT_IMPLEMENTED`, or `ESP_ERR_NOT_SUPPORTED` on the selected path keeps the physical gate red.


## 10. Gateway Edge Authority
### `firmware/gateway/main/policy_guard.h`


In [ ]:
%%writefile /content/cldt_scratch/policy_guard.h
#ifndef CLDT_GATEWAY_POLICY_GUARD_H
#define CLDT_GATEWAY_POLICY_GUARD_H

#include <stdbool.h>
#include <stdint.h>

#include "cldt/cldt_control_profile.h"
#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    /*
     * Effective per-run ceilings are derived from the resolved control profile
     * and capped by compiled safety maxima. A manifest cannot raise them.
     */
    uint32_t maximum_total_rate_pps;
    uint32_t minimum_critical_period_ms;
    uint16_t maximum_bulk_burst_packets;
    uint32_t maximum_policy_ttl_ms;
} cldt_edge_limits_t;

typedef struct {
    /* Only this object owns the mutable edge policy snapshot and applied epoch. */
    cldt_run_id_t active_run_id;
    cldt_boot_id_t command_authority_boot_id;
    cldt_policy_epoch_t applied_epoch;
    /*
     * Archived with every decision so a gateway trace can identify the exact
     * resolved profile even if a human-readable profile name is later reused.
     */
    uint8_t control_profile_digest[CLDT_CONTROL_PROFILE_DIGEST_BYTES];
    cldt_policy_t safe_fallback;
    cldt_policy_t active_policy;
    cldt_edge_limits_t limits;
    bool remote_actuation_enabled;
} cldt_policy_guard_t;

cldt_status_t cldt_policy_guard_init(
    cldt_policy_guard_t *guard,
    const cldt_control_profile_t *control_profile,
    const cldt_edge_limits_t *limits,
    const cldt_policy_t *safe_fallback);

/*
 * Binds the guard to one nonzero run already reserved in the global run ledger
 * and resets it to the safe policy. An actuated run is additionally bound to
 * the command-key identity. A gateway boot never resumes forwarding an old run.
 * Remote actuation remains fail-closed unless both the build maturity switch
 * and the frozen ready manifest authorize it.
 */
cldt_status_t cldt_policy_guard_begin_run(
    cldt_policy_guard_t *guard,
    cldt_run_id_t run_id,
    cldt_boot_id_t command_authority_boot_id,
    bool remote_actuation_requested);

/*
 * Validates host output independently; it does not trust the host gate state.
 * Caller supplies local time, uncertainty, and health observed at acceptance.
 * On failure, the function must leave the active policy and epoch unchanged.
 */
cldt_status_t cldt_policy_guard_accept(
    cldt_policy_guard_t *guard,
    cldt_run_id_t proposal_run_id,
    cldt_boot_id_t proposal_authority_boot_id,
    const cldt_policy_t *proposal,
    uint64_t now_gateway_us,
    uint32_t clock_uncertainty_us,
    bool local_health_ok);

/*
 * Immediately selects the compiled safe policy and records the reason. This
 * must be usable while host, broker, or model connectivity is absent.
 */
cldt_status_t cldt_policy_guard_fallback(
    cldt_policy_guard_t *guard,
    cldt_status_t reason);

/* Disarms remote control, restores fallback, and clears run/epoch after trace. */
cldt_status_t cldt_policy_guard_end_run(
    cldt_policy_guard_t *guard,
    cldt_status_t terminal_reason);

#ifdef __cplusplus
}
#endif

#endif


### `firmware/gateway/main/policy_guard.c`


In [ ]:
%%writefile /content/cldt_scratch/policy_guard.c
#include "policy_guard.h"

cldt_status_t cldt_policy_guard_init(
    cldt_policy_guard_t *guard,
    const cldt_control_profile_t *control_profile,
    const cldt_edge_limits_t *limits,
    const cldt_policy_t *safe_fallback)
{
    (void)guard;
    (void)control_profile;
    (void)limits;
    (void)safe_fallback;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject null arguments and validate the resolved control profile before
     *    accepting any edge limits. The caller must calculate limits by taking
     *    the stricter value of profile settings and build-time safety maxima.
     * 2. Copy the profile digest, effective limits, and safe fallback into
     *    guard-owned storage. Validate fallback with the same arithmetic used
     *    for host proposals; a fallback may never violate the effective limits.
     * 3. Set active policy to the safe fallback, clear active run, command
     *    authority boot ID, and applied epoch, and begin with
     *    remote_actuation_enabled false. A failed
     *    initialization must leave no policy that could be mistaken for an
     *    accepted remote command.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_begin_run(
    cldt_policy_guard_t *guard,
    cldt_run_id_t run_id,
    cldt_boot_id_t command_authority_boot_id,
    bool remote_actuation_requested)
{
    (void)guard;
    (void)run_id;
    (void)command_authority_boot_id;
    (void)remote_actuation_requested;

    /*
     * IMPLEMENTATION TODO: require an initialized guard, nonzero run ID, no
     * active run, a nonzero command_authority_boot_id, and admission evidence
     * that the host reserved this ID in the durable global run ledger. If remote
     * actuation is requested, also require its binding to the active command-key
     * identity. A fresh gateway boot must refuse to resume forwarding an old
     * run. Only after that uniqueness boundary is proven may the guard restore
     * fallback, reset applied_epoch, and bind run_id plus command authority.
     * Enable remote acceptance only when requested by the frozen ready manifest
     * and CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION is enabled; otherwise keep the run
     * shadow-only. Trace the resulting mode.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_accept(
    cldt_policy_guard_t *guard,
    cldt_run_id_t proposal_run_id,
    cldt_boot_id_t proposal_authority_boot_id,
    const cldt_policy_t *proposal,
    uint64_t now_gateway_us,
    uint32_t clock_uncertainty_us,
    bool local_health_ok)
{
    (void)guard;
    (void)proposal_run_id;
    (void)proposal_authority_boot_id;
    (void)proposal;
    (void)now_gateway_us;
    (void)clock_uncertainty_us;
    (void)local_health_ok;

    /*
     * IMPLEMENTATION TODO: hold the short policy critical section only while
     * checking remote_actuation_enabled, local health, exact proposal_run_id and
     * proposal_authority_boot_id equality with the commissioned run authority,
     * strictly increasing epoch, finite TTL, clock
     * uncertainty, critical-period protection, bulk burst limit, and total rate.
     * Validate the full proposal before swapping it. On any failure leave the
     * active policy and epoch unchanged, return a reason code, and ensure the
     * caller emits a rejection trace/acknowledgement outside the lock. On
     * success the caller forwards the retained authenticated datagram byte for
     * byte; it must not re-encode different bytes under the accepted nonce.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_end_run(
    cldt_policy_guard_t *guard,
    cldt_status_t terminal_reason)
{
    (void)guard;
    (void)terminal_reason;

    /*
     * IMPLEMENTATION TODO: disarm remote acceptance first, atomically restore
     * the safe policy, emit the final policy/epoch/run/authority record, then
     * clear the active run, command authority, and applied epoch. Never clear
     * identity before the terminal trace is durable enough for the gateway's
     * evidence path.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_policy_guard_fallback(
    cldt_policy_guard_t *guard,
    cldt_status_t reason)
{
    (void)guard;
    (void)reason;

    /*
     * IMPLEMENTATION TODO: atomically replace active_policy with the compiled
     * safe snapshot and record the supplied failure reason and current accepted
     * epoch in a traceable event. Retain active_run_id and applied_epoch so a
     * later requalified command must still advance strictly. Stop forwarding
     * new proposals. Endpoints return to their own compiled safe policy when
     * the last accepted finite command
     * expires; version one must not synthesize a second command under the host's
     * nonce space. Fallback must succeed without host, broker, or model access.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


The guard is the sole mutable owner of active edge policy and epoch.

1. `cldt_policy_guard_init()` validates profile, effective limits, and safe fallback; remote actuation begins false.
2. `cldt_policy_guard_begin_run()` binds a newly reserved run and coordinator authority. The compile switch and ready manifest are both required, but neither bypasses command authentication.
3. `cldt_policy_guard_accept()` receives only a command already decoded/authenticated by the gateway command owner. Inside one short critical section it checks mode, local health, exact run/authority, strictly newer epoch, finite TTL, uncertainty, critical protection, burst, and total rate.
4. Any rejection leaves policy and epoch byte-for-byte unchanged. Trace/ACK occurs after lock release and preserves the exact status.
5. Acceptance swaps one complete immutable snapshot, records epoch, and forwards the retained original datagram. Re-encoding is not allowed.
6. `cldt_policy_guard_fallback()` restores gateway-safe policy without host/broker/model access and retains run/epoch for audit and monotonic requalification.
7. The header/source must state what re-enables proposal acceptance after fallback. A fresh clean host sequence may requalify within the same run only through one explicit supervisor transition and a strictly higher epoch; otherwise a new run is required.
8. `cldt_policy_guard_end_run()` disarms first, records final identity, then clears state.

The gateway fallback does not magically rewrite endpoint policy. Endpoints independently return to their compiled safe policy by finite TTL unless a separately designed revocation protocol exists; version one has none.


### Gateway Runtime
#### `firmware/gateway/main/gateway_runtime.h`


In [ ]:
%%writefile /content/cldt_scratch/gateway_runtime.h
#ifndef CLDT_GATEWAY_RUNTIME_H
#define CLDT_GATEWAY_RUNTIME_H

#include <stdbool.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/event_groups.h"
#include "freertos/queue.h"
#include "freertos/task.h"

#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_types.h"
#include "policy_guard.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef enum {
    CLDT_GATEWAY_BOOT = 0,
    CLDT_GATEWAY_PROVISIONING,
    CLDT_GATEWAY_FORMING_THREAD,
    CLDT_GATEWAY_IDLE,
    CLDT_GATEWAY_WARMUP,
    CLDT_GATEWAY_MEASURING,
    CLDT_GATEWAY_COOLDOWN,
    CLDT_GATEWAY_FALLBACK,
    CLDT_GATEWAY_FAULT
} cldt_gateway_state_t;

typedef struct {
    cldt_node_id_t node_id;
    cldt_gateway_state_t state;
    /* Sole owner of active run, policy, epoch, and remote-actuation state. */
    cldt_policy_guard_t policy_guard;
    QueueHandle_t thread_rx_queue;
    QueueHandle_t observation_queue;
    QueueHandle_t command_queue;
    EventGroupHandle_t events;
    TaskHandle_t supervisor_task;
    TaskHandle_t aggregator_task;
    TaskHandle_t publisher_task;
    bool started;
} cldt_gateway_runtime_t;

/*
 * Initializes caller-owned state and all static RTOS objects. No task may run
 * and no radio may start before this function succeeds completely.
 */
esp_err_t cldt_gateway_runtime_init(cldt_gateway_runtime_t *runtime);

/* Starts tasks only after provisioning, RCP, Thread, and backhaul are ready. */
esp_err_t cldt_gateway_runtime_start(cldt_gateway_runtime_t *runtime);

/* Requests bounded shutdown at a run boundary; it must not delete live tasks. */
esp_err_t cldt_gateway_runtime_request_stop(cldt_gateway_runtime_t *runtime);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/gateway_runtime.c`


In [ ]:
%%writefile /content/cldt_scratch/gateway_runtime.c
#include "gateway_runtime.h"

esp_err_t cldt_gateway_runtime_init(cldt_gateway_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: reject a null runtime, clear state, load gateway node
     * identity, initialize the sole policy guard with its compiled safe policy,
     * and create all queues, event groups, trace storage, timer, and task stacks
     * statically. A new gateway boot has no resumable command-forwarding run;
     * remote actuation remains disarmed until a newly ledger-reserved run is
     * admitted.
     * Choose queue lengths from measured producer rates and document each owner.
     * Do not start Thread, Wi-Fi, BLE, broker, or any task here. An initialization
     * failure must leave the device in BOOT with no partially live project task.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_runtime_start(cldt_gateway_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: require successful init and all external prerequisites
     * (provisioning, RCP, Thread attach, backhaul readiness as required), start
     * supervisor first, verify admission of a newly ledger-reserved run, bind the
     * guard to it, then start aggregator and publisher under supervisor control.
     * Never resume authenticated command forwarding for a pre-reboot run. Give
     * each task a narrow ownership contract and measure stack margin before
     * choosing final priority/core affinity. If a later task fails, supervisor
     * stops earlier tasks in reverse order and emits a fault record rather than
     * continuing half-up.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_runtime_request_stop(cldt_gateway_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: set a supervisor event or task notification only. The
     * supervisor must stop manifest admission, command forwarding, and periodic
     * publication in a defined sequence, then request final endpoint counters and
     * emit final gateway status. Direct vTaskDelete from a caller would bypass
     * queue ownership and make accounting loss impossible to diagnose.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


The generic `command_queue` needs a bounded item contract before task creation. One item retains immutable raw bytes, receive time, broker retained/duplicate metadata, decoded identity, and later gateway/two-endpoint outcomes. The queue never stores borrowed MQTT or OpenThread pointers.

Supervisor ordering:

1. Init creates static objects and a disarmed guard; no network/task side effect.
2. Start verifies provisioning, RCP/Thread, backhaul, newly commissioned run, and command-key identity.
3. Command consumer rejects retained, fragmented-incomplete, oversize, wrong-topic, unauthenticated, wrong-run, and wrong-authority input before guard.
4. Accepted raw bytes are fanned to the two admitted endpoint IPv6 identities with bounded ACK tracking.
5. Host/broker loss, local health failure, expiry, and injected observation pause generate explicit events and cannot leave a hidden half-active policy.
6. Stop closes manifest admission and command intake, resolves pending ACK/expiry state, captures final counters, ends guard, then tears down transport.


### Gateway Backhaul
#### `firmware/gateway/main/backhaul.h`


In [ ]:
%%writefile /content/cldt_scratch/backhaul.h
#ifndef CLDT_GATEWAY_BACKHAUL_H
#define CLDT_GATEWAY_BACKHAUL_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#include "cldt/cldt_types.h"
#include "gateway_provisioning.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_backhaul_command_fn)(
    void *context,
    const uint8_t *payload,
    size_t payload_bytes,
    bool retained,
    uint64_t received_local_us);

typedef struct {
    const cldt_gateway_credentials_t *credentials;
    cldt_backhaul_command_fn on_command;
    void *callback_context;
} cldt_backhaul_config_t;

esp_err_t cldt_backhaul_init(const cldt_backhaul_config_t *config);
esp_err_t cldt_backhaul_start(void);

/* Publishes immutable observation bytes; caller retains ownership. */
esp_err_t cldt_backhaul_publish_observation(
    cldt_run_id_t run_id,
    cldt_node_id_t node_id,
    const uint8_t *payload,
    size_t payload_bytes);

/* Starts a local-only HTTP endpoint for a pending manifest, never telemetry. */
esp_err_t cldt_backhaul_start_manifest_server(void);

esp_err_t cldt_backhaul_stop(void);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/backhaul.c`


In [ ]:
%%writefile /content/cldt_scratch/backhaul.c
#include "backhaul.h"

esp_err_t cldt_backhaul_init(const cldt_backhaul_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate credentials and callbacks without logging
     * secrets, initialize Wi-Fi station mode, wait for private-LAN readiness,
     * create MQTT client state, and register callbacks that only enqueue bounded
     * command or connection events. Do not start a measured run, publish a policy,
     * or start the HTTP server from init. Every partial resource needs a defined
     * cleanup path for a failed credential, Wi-Fi, or broker connection.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_start(void)
{
    /*
     * IMPLEMENTATION TODO: start Wi-Fi and broker connections through an explicit
     * state machine, publish connection/health changes as traceable events, and
     * use bounded backoff outside any Thread, policy, or timing-critical lock.
     * A reconnect must never resurrect an expired policy or turn a retained broker
     * message into a command. The local guard remains safe while backhaul is down.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_publish_observation(
    cldt_run_id_t run_id,
    cldt_node_id_t node_id,
    const uint8_t *payload,
    size_t payload_bytes)
{
    (void)run_id;
    (void)node_id;
    (void)payload;
    (void)payload_bytes;

    /*
     * IMPLEMENTATION TODO: validate run/node IDs and bounded payload size, add
     * immutable envelope metadata, publish observations at the selected QoS, and
     * make disconnected behavior explicit: either bounded drop with a trace event
     * or a bounded local queue with a recorded high-water mark. Never block a
     * Thread receive task on broker reconnect and never use retained observations
     * as current truth for a controller.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_start_manifest_server(void)
{
    /*
     * IMPLEMENTATION TODO: bind only to the private experiment interface, cap
     * request/body size, accept a single complete manifest document into staging
     * storage, validate syntax and schema before acknowledgement, and pass only a
     * digest plus approved subset to supervisor. The HTTP handler may never alter
     * an active run. Reject credentials, command injection, and high-rate uploads.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_backhaul_stop(void)
{
    /*
     * IMPLEMENTATION TODO: stop accepting HTTP uploads first, make command
     * callbacks reject new policy input, signal the supervisor that backhaul is
     * leaving service, drain or account for only the bounded observation queue,
     * disconnect MQTT and Wi-Fi with bounded timeouts, and clear callback state.
     * The function must be safe after a partial init and must not overwrite run
     * evidence just because the network is unavailable.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


ESP-MQTT can deliver retained/duplicate flags and may split one application message across multiple data events. The callback therefore assembles only one bounded complete command, records metadata, and enqueues owned bytes; it never calls the guard directly. Reference: [ESP-MQTT event contract](https://docs.espressif.com/projects/esp-idf/en/stable/esp32/api-reference/protocols/mqtt.html).

Week-5 command rules:

1. Exact command topic/QoS/run binding is frozen with the broker contract.
2. Retained commands are always rejected. A reconnect cannot resurrect an expired optimization.
3. Duplicate delivery preserves raw evidence but a previously accepted epoch cannot apply twice.
4. Fragment assembly is bounded by exact expected command size. Topic/data fragments from separate messages never mix.
5. Observation pause affects only the named publication boundary; command, Thread, endpoint traffic, and topology remain unchanged.
6. Backhaul failure leaves the guard and endpoints on local safety behavior; reconnect does not re-arm control by itself.


### Immutable Thread Fan-Out
#### `firmware/gateway/main/thread_bridge.h`


In [ ]:
%%writefile /content/cldt_scratch/thread_bridge.h
#ifndef CLDT_GATEWAY_THREAD_BRIDGE_H
#define CLDT_GATEWAY_THREAD_BRIDGE_H

#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_thread_frame_fn)(
    void *context,
    const uint8_t *datagram,
    size_t datagram_bytes,
    const uint8_t source_ipv6[16],
    int8_t rssi_dbm,
    uint64_t received_local_us);

typedef struct {
    cldt_thread_frame_fn on_frame;
    void *callback_context;
    uint16_t listen_port;
} cldt_thread_bridge_config_t;

/* Attaches the project UDP adapter after the upstream border router is ready. */
esp_err_t cldt_thread_bridge_init(const cldt_thread_bridge_config_t *config);

esp_err_t cldt_thread_bridge_send(
    const uint8_t destination_ipv6[16],
    const uint8_t *datagram,
    size_t datagram_bytes);

/* Captures current role/partition/neighbor state into caller-owned output. */
esp_err_t cldt_thread_bridge_snapshot(
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/thread_bridge.c`


In [ ]:
%%writefile /content/cldt_scratch/thread_bridge.c
#include "thread_bridge.h"

esp_err_t cldt_thread_bridge_init(const cldt_thread_bridge_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate callback, context, and listen port; confirm
     * the upstream border router and RCP are initialized and attached; then bind
     * one project UDP socket. The receive callback must copy or enqueue a bounded
     * datagram plus source metadata and return promptly. Do not retain pointers
     * owned by OpenThread, and make RCP/Thread detach visible to gateway runtime.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_bridge_send(
    const uint8_t destination_ipv6[16],
    const uint8_t *datagram,
    size_t datagram_bytes)
{
    (void)destination_ipv6;
    (void)datagram;
    (void)datagram_bytes;

    /*
     * IMPLEMENTATION TODO: require a non-null 16-byte destination and one bounded
     * datagram, verify protocol size before taking any OpenThread API lock, send
     * exactly one datagram, and return a precise local send status. This function
     * consumes no caller buffer ownership; caller may reuse its memory after return
     * only if the upstream API has copied it. Document that behavior explicitly.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_bridge_snapshot(
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes)
{
    (void)output;
    (void)output_capacity;
    (void)output_bytes;

    /*
     * IMPLEMENTATION TODO: require output/output_bytes, acquire the OpenThread
     * lock briefly, serialize only the selected current role, partition, parent,
     * and neighbor/link fields into caller-owned bytes, then release the lock.
     * Bound every list and output length; a partial snapshot must be marked as
     * partial rather than presented as complete topology evidence.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


The global command uses one authenticated byte array. Gateway fan-out sends those exact bytes to endpoint A and B addresses resolved from the admitted run; an endpoint selector is not inserted into the command.

1. Address mapping is frozen before the run and rejects unknown/duplicate labels.
2. One send completion is local transport evidence, not endpoint acceptance.
3. Each endpoint returns a separate attributable acknowledgement record under the fixed project-frame and evidence contract; this notebook does not silently extend command authentication semantics to acknowledgements.
4. Partial fan-out, one missing ACK, role/partition change, or detach remains visible and prevents global-success classification.
5. OpenThread locks protect only API access. Broker wait, ACK deadline, recorder I/O, and guard lock remain outside.


### Gateway Provisioning
#### `firmware/gateway/main/gateway_provisioning.h`


In [ ]:
%%writefile /content/cldt_scratch/gateway_provisioning.h
#ifndef CLDT_GATEWAY_PROVISIONING_H
#define CLDT_GATEWAY_PROVISIONING_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_WIFI_SSID_MAX_BYTES 32U
#define CLDT_WIFI_PASSPHRASE_MAX_BYTES 64U
#define CLDT_BROKER_URI_MAX_BYTES 128U
#define CLDT_COMMAND_KEY_BYTES 32U

typedef struct {
    char wifi_ssid[CLDT_WIFI_SSID_MAX_BYTES + 1U];
    char wifi_passphrase[CLDT_WIFI_PASSPHRASE_MAX_BYTES + 1U];
    char broker_uri[CLDT_BROKER_URI_MAX_BYTES];
    uint8_t command_key[CLDT_COMMAND_KEY_BYTES];
    uint32_t node_id;
} cldt_gateway_credentials_t;

/* Loads validated credentials from encrypted/protected NVS where available. */
esp_err_t cldt_gateway_provisioning_load(
    cldt_gateway_credentials_t *output,
    bool *is_provisioned);

/*
 * Runs a temporary authenticated BLE GATT service. The implementation must
 * require physical presence, bound every characteristic, and stop advertising
 * before a measured run begins.
 */
esp_err_t cldt_gateway_provisioning_start_ble(void);

esp_err_t cldt_gateway_provisioning_stop_ble(void);

/* Erasure must require a deliberate local action; never expose it over MQTT. */
esp_err_t cldt_gateway_provisioning_erase(void);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/gateway/main/gateway_provisioning.c`


In [ ]:
%%writefile /content/cldt_scratch/gateway_provisioning.c
#include "gateway_provisioning.h"

esp_err_t cldt_gateway_provisioning_load(
    cldt_gateway_credentials_t *output,
    bool *is_provisioned)
{
    (void)output;
    (void)is_provisioned;

    /*
     * IMPLEMENTATION TODO: initialize the approved NVS namespace, read only
     * project-owned keys into bounded temporary buffers, validate lengths and
     * NUL termination before copying to output, and report provisioned false for
     * incomplete data. Never print SSID, passphrase, broker URI credentials, or
     * command key. Zero temporary secret buffers on every error and success path.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_provisioning_start_ble(void)
{
    /*
     * IMPLEMENTATION TODO: expose only the characteristics needed to provision
     * gateway identity, Wi-Fi credentials, broker endpoint, and command trust
     * material; require explicit local physical presence before advertising; and
     * use authenticated pairing or a documented secure enrollment procedure. Bound
     * every write length, reject reads of secrets, and persist only after complete
     * validation. BLE is provisioning-only and never carries measured telemetry.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_provisioning_stop_ble(void)
{
    /*
     * IMPLEMENTATION TODO: stop advertising and GATT service, disconnect active
     * clients, zero temporary pairing and credential buffers, release NimBLE
     * resources as required by ESP-IDF, and emit only a non-sensitive state
     * transition. Require this operation before a measured run so BLE coexistence
     * cannot become an undocumented 2.4 GHz treatment variable.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_gateway_provisioning_erase(void)
{
    /*
     * IMPLEMENTATION TODO: sample a designated physical button with debounce and
     * a long-press confirmation window, visibly indicate pending erase without
     * exposing secrets, delete only the project's NVS namespace, zero in-memory
     * copies, and reboot into unprovisioned state. MQTT, HTTP, and BLE must never
     * be able to invoke this operation remotely.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


The current gateway credential struct includes the 32-byte command key; endpoint provisioning has no equivalent owner yet. Gateway work is complete only when:

1. Secret material enters through physical-presence enrollment and never appears in log/evidence/Git.
2. Ordinary unencrypted NVS is not called protected storage. The chosen local secret mechanism and its limitation are documented.
3. Only non-secret key identity is archived with the run.
4. BLE/provisioning and HTTP staging are fully stopped before measurement so coexistence is not an undocumented treatment.
5. Erase remains local and cannot be triggered by MQTT/HTTP/BLE command input.
6. Gateway, host, A, and B fixed-vector parity is proven before the group key is used physically.


### Gateway Entry Point
#### `firmware/gateway/main/app_main.c`


In [ ]:
%%writefile /content/cldt_scratch/gateway_app_main.c
#include "esp_log.h"

static const char *TAG = "cldt_gateway";

void app_main(void)
{
    ESP_LOGW(TAG,
             "Research scaffold only: gateway runtime, provisioning, Thread "
             "bridge, backhaul, and policy guard are not implemented.");

    /*
     * IMPLEMENTATION ORDER:
     * 1. Prove provisioning storage and physical erase behavior without a run.
     * 2. Bring up the upstream RCP and Thread border router with no project policy.
     * 3. Add bounded Thread bridging and append-only observation publication.
     * 4. Add local policy guard, then only one expiring bulk-rate action.
     * 5. Compare SMP and unicore only after identical-build evidence exists.
     * Keep this entry point thin: construct runtime, call init/start, and let the
     * supervisor own failure handling. It must never become an ad-hoc demo flow.
     */
}


The entry point remains thin exactly as its implementation-order comment requires. It constructs the single `cldt_gateway_runtime_t`, calls init/start once, and reports a terminal startup failure; it does not parse command bytes, call `cldt_policy_guard_accept()` directly, wait for ACKs, or implement a second fallback path. The supervisor remains the lifecycle owner. A safety-capable image that still prints the scaffold warning and returns without starting the runtime has not reached the hardware gate.


### Gateway Project Build Root
#### `firmware/gateway/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/gateway_project_CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(EXTRA_COMPONENT_DIRS "${CMAKE_CURRENT_LIST_DIR}/../../common")

include($ENV{IDF_PATH}/tools/cmake/project.cmake)
project(cldt_gateway)


This project root already imports `common` through `EXTRA_COMPONENT_DIRS`; Week 5 does not create a second common copy. Component sources and ESP-IDF dependencies stay in `firmware/gateway/main/CMakeLists.txt`. The shadow and safety configurations use separate fresh build directories so a stale cache cannot make `CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION` or crypto options appear in the wrong binary. The archived project name, IDF revision, complete `sdkconfig`, partition table, and artifact hashes must resolve to the flashed image.


### Gateway Build and Maturity Switch
#### `firmware/gateway/main/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/gateway_CMakeLists.txt
idf_component_register(
    SRCS
        "app_main.c"
        "gateway_runtime.c"
        "gateway_provisioning.c"
        "thread_bridge.c"
        "backhaul.c"
        "policy_guard.c"
    INCLUDE_DIRS "."
    REQUIRES
        common
        freertos
        nvs_flash
        esp_event
        esp_netif
        esp_wifi
        esp_timer
        bt
        mqtt
        esp_http_server
        openthread
)


#### `firmware/gateway/main/Kconfig.projbuild`


In [ ]:
%%writefile /content/cldt_scratch/gateway_Kconfig.projbuild
menu "CLDT Gateway"

config CLDT_GATEWAY_NODE_ID
    int "Provisioning fallback node ID"
    range 1 4294967295
    default 1
    help
        Used only before a provisioned identity is present in NVS.

config CLDT_GATEWAY_RX_QUEUE_LENGTH
    int "Thread receive queue slots"
    range 4 128
    default 32
    help
        Bounded slots between Thread receive and aggregation. Select using peak
        observed arrival rate and consumer service time, then trace high-water.

config CLDT_GATEWAY_TRACE_QUEUE_LENGTH
    int "Observation queue slots"
    range 16 512
    default 128
    help
        Bounded observation queue. Overflow must become a visible health event;
        this setting is not permission to retain arbitrary broker backlog.

config CLDT_GATEWAY_MAX_TOTAL_RATE_PPS
    int "Compiled maximum aggregate application rate"
    range 1 500
    default 100
    help
        Compiled aggregate ceiling enforced by the edge guard. It is independent
        of host prediction and remains active during a host or broker failure.

config CLDT_GATEWAY_MAX_POLICY_TTL_MS
    int "Compiled maximum policy lifetime"
    range 1000 60000
    default 10000
    help
        Longest acceptable remote-policy lifetime. A control_profile may request
        a shorter TTL but cannot extend a command beyond this local bound.

config CLDT_GATEWAY_REMOTE_ACTUATION
    bool "Compile candidate remote-actuation path"
    default n
    help
        Keep disabled during testbed and shadow-model work. Enabling this switch
        does not bypass ready-manifest, authentication, run/epoch, TTL, local
        health, edge-limit, fallback, or endpoint validation requirements.

config CLDT_GATEWAY_RCP_UART_RX_GPIO
    int "RCP UART receive GPIO"
    range 0 48
    default 18
    help
        Gateway receive pin connected to the RCP transmit pin. Verify crossover,
        voltage compatibility, and the selected RCP UART configuration together.

config CLDT_GATEWAY_RCP_UART_TX_GPIO
    int "RCP UART transmit GPIO"
    range 0 48
    default 17
    help
        Gateway transmit pin connected to the RCP receive pin. Record final
        wiring, baud rate, and reset behavior in the upstream bring-up evidence.

config CLDT_GATEWAY_RCP_RESET_GPIO
    int "RCP reset GPIO"
    range -1 48
    default -1
    help
        -1 leaves optional RCP reset control disabled. Select a free S3 GPIO only
        after the C6 EN electrical behavior and wiring have been verified.

config CLDT_GATEWAY_RCP_BOOT_GPIO
    int "RCP boot GPIO"
    range -1 48
    default -1
    help
        -1 leaves optional RCP boot-mode control disabled. Treat any enabled
        value as board-specific wiring and validate it before automated reset.

endmenu


#### `firmware/gateway/sdkconfig.defaults`


In [ ]:
%%writefile /content/cldt_scratch/gateway_sdkconfig.defaults
CONFIG_FREERTOS_HZ=1000
CONFIG_FREERTOS_USE_TRACE_FACILITY=y
CONFIG_FREERTOS_GENERATE_RUN_TIME_STATS=y
CONFIG_ESP_TASK_WDT_EN=y
CONFIG_BT_ENABLED=y
CONFIG_BT_NIMBLE_ENABLED=y
CONFIG_OPENTHREAD_ENABLED=y
CONFIG_MQTT_PROTOCOL_311=y
CONFIG_MBEDTLS_CHACHAPOLY_C=y
CONFIG_MBEDTLS_CHACHA20_C=y
CONFIG_MBEDTLS_POLY1305_C=y


A safety-capable image is built in a fresh directory and archived separately from the shadow image. `CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION` remains disabled through deterministic tests and only becomes enabled in the exact physical safety binary after all admission checks pass. Enabling it does not prove authorization.

| Gateway build evidence | Example format |
|---|---|
| Source / ESP-IDF revision | e.g. commit IDs |
| Full sdkconfig | e.g. archived file + digest |
| Binary/partition/flasher identities | e.g. SHA-256 per artifact |
| Remote-actuation switch | e.g. `y` only in admitted safety image |
| Effective rate/TTL limits | e.g. Kconfig vs profile minimum |
| Queue length/high-water | e.g. configured / observed |
| Provisioning stopped before measure | e.g. timestamped state trace |


## 11. Endpoint Local Authority
### `firmware/endpoint/main/endpoint_runtime.h`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_runtime.h
#ifndef CLDT_ENDPOINT_RUNTIME_H
#define CLDT_ENDPOINT_RUNTIME_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/event_groups.h"
#include "freertos/task.h"

#include "cldt/cldt_clock_sync.h"
#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_types.h"
#include "deadline_queue.h"
#include "power_probe.h"
#include "workload.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef enum {
    CLDT_ENDPOINT_BOOT = 0,
    CLDT_ENDPOINT_COMMISSIONING,
    CLDT_ENDPOINT_ATTACHED,
    CLDT_ENDPOINT_IDLE,
    CLDT_ENDPOINT_RUNNING,
    CLDT_ENDPOINT_FALLBACK,
    CLDT_ENDPOINT_FAULT
} cldt_endpoint_state_t;

typedef struct {
    cldt_node_id_t node_id;
    cldt_node_role_t role;
    cldt_endpoint_state_t state;
    /* RAM mirrors loaded from an integrity-checked durable replay record. */
    cldt_run_id_t active_run_id;
    cldt_boot_id_t command_authority_boot_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t next_sequence;
    cldt_policy_epoch_t applied_epoch;
    cldt_policy_t safe_policy;
    cldt_policy_t active_policy;
    cldt_clock_sync_t clock_sync;
    cldt_deadline_queue_t deadline_queue;
    cldt_workload_t workload;
    cldt_event_trace_t trace;
    EventGroupHandle_t events;
    TaskHandle_t supervisor_task;
    TaskHandle_t transmitter_task;
    TaskHandle_t trace_task;
    TaskHandle_t power_task;
    /* False forbids remote apply and keeps the compiled safe policy active. */
    bool replay_state_valid;
    bool started;
} cldt_endpoint_runtime_t;

esp_err_t cldt_endpoint_runtime_init(cldt_endpoint_runtime_t *runtime);
esp_err_t cldt_endpoint_runtime_start(cldt_endpoint_runtime_t *runtime);

/* Validates coordinator identity, run, durable epoch, TTL, and local limits. */
esp_err_t cldt_endpoint_runtime_receive_command(
    cldt_endpoint_runtime_t *runtime,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us);

esp_err_t cldt_endpoint_runtime_request_stop(cldt_endpoint_runtime_t *runtime);

#ifdef __cplusplus
}
#endif

#endif


### `firmware/endpoint/main/endpoint_runtime.c`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_runtime.c
#include "endpoint_runtime.h"

esp_err_t cldt_endpoint_runtime_init(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: reject a null runtime, clear caller-owned state, load
     * immutable board identity and role, generate a boot ID that changes after a
     * reset, and load the integrity-checked durable replay record containing the
     * enrolled run, coordinator boot identity, and highest accepted epoch. A
     * missing or corrupt record leaves
     * replay_state_valid false: retain the compiled safe policy and require an
     * explicitly commissioned new unique run before remote apply. Create every
     * steady-state queue, trace buffer, event group, and task storage statically
     * and leave the state at BOOT. No radio attach, workload release, or dynamic
     * allocation is permitted here. Fail before changing externally visible
     * state on any error.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_start(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: require successful initialization, then start the
     * supervisor task first. It owns transitions through commissioning, attach,
     * idle, running, fallback, and fault. Start transport, workload, trace, and
     * optional power tasks only after the supervisor reports their prerequisites;
     * if any task creation fails, notify supervisor to unwind already started
     * components. Do not start release timers merely because Thread attached.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_receive_command(
    cldt_endpoint_runtime_t *runtime,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us)
{
    (void)runtime;
    (void)datagram;
    (void)datagram_bytes;
    (void)received_local_us;

    /*
     * IMPLEMENTATION TODO:
     * 1. Copy or retain the datagram only for the duration required by the
     *    decoder; reject oversized input before queueing work.
     * 2. Decode and authenticate it; require coordinator authority node ID 0 and
     *    the coordinator boot/session identity commissioned for this run, the
     *    enrolled run ID, a valid durable replay state, a strictly newer epoch,
     *    a live TTL, and endpoint-local limits. Map received_local_us into the
     *    gateway monotonic domain through the validated clock-sync state and
     *    reject excessive uncertainty; never compare unrelated local clocks.
     *    The command boot ID identifies the coordinator process, not this
     *    endpoint and not replay state. A host decision is not local authorization.
     * 3. Atomically persist the new (run_id, coordinator_boot_id, highest_epoch)
     *    before publishing
     *    one immutable policy snapshot at a workload release boundary. If the
     *    durable write fails, reject and retain the safe policy. A duplicate
     *    accepted epoch must be acknowledged as duplicate, never applied twice.
     * 4. Emit a trace record and an acknowledgement for every accept or reject
     *    reason. On missing/corrupt replay state or any ambiguity, preserve the
     *    safe policy, enter FALLBACK, and require a newly commissioned run.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_endpoint_runtime_request_stop(cldt_endpoint_runtime_t *runtime)
{
    (void)runtime;

    /*
     * IMPLEMENTATION TODO: request a supervisor-owned stop, block new workload
     * releases, let producer and transport finish or explicitly expire queued
     * work, request final counters, and reconcile before changing state to IDLE.
     * A stopped endpoint must retain its safe policy and remain able to report
     * health. Do not delete a task from an arbitrary caller or discard evidence
     * merely to make shutdown appear fast.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


This runtime owns the final apply decision, but its current header lacks command authentication and durable-store ownership. Those contracts close before `receive_command()`.

1. Boot loads immutable node/role, creates a new endpoint `boot_id`, initializes safe policy, loads command key through the approved secret mechanism, and validates one versioned replay record.
2. The replay record covers enrolled `run_id`, commissioned coordinator boot ID, and highest accepted epoch plus integrity/version data. A missing/corrupt record sets `replay_state_valid == false`, keeps safe policy, and requires a new run.
3. Command bytes are copied into bounded owned storage, decoded, authenticated, and checked for authority node 0, commissioned boot, run, durable epoch, TTL, clock uncertainty, and local limits.
4. Persist-before-apply uses one atomic record update, explicit commit, and readback verification before publishing policy. ESP-IDF documents that NVS updates require `nvs_commit()`; NVS is designed for power-loss-resistant key-value updates, while a write in progress may leave the old rather than the new value. Reference: [ESP-IDF NVS](https://docs.espressif.com/projects/esp-idf/en/stable/esp32/api-reference/storage/nvs_flash.html).
5. Failed write/readback returns rejection and preserves safe policy. Conservative loss of a new command is acceptable; repeating an applied command is not.
6. Apply occurs at one workload release boundary and produces ACK/reject evidence with exact reason.
7. A supervisor-owned expiry check returns `active_policy` to `safe_policy` no later than finite TTL, even without host/gateway access.
8. Endpoint restart changes endpoint boot evidence but not the valid durable replay state.


### Atomic Workload Apply
#### `firmware/endpoint/main/workload.h`


In [ ]:
%%writefile /content/cldt_scratch/workload.h
#ifndef CLDT_ENDPOINT_WORKLOAD_H
#define CLDT_ENDPOINT_WORKLOAD_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"
#include "freertos/timers.h"

#include "cldt/cldt_types.h"
#include "deadline_queue.h"

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_ENDPOINT_MAX_STREAMS 8U

typedef struct {
    uint32_t stream_id;
    cldt_traffic_class_t traffic_class;
    uint32_t period_ms;
    uint32_t phase_ms;
    uint32_t jitter_ms;
    uint32_t deadline_ms;
    uint16_t payload_bytes;
    uint16_t burst_packets;
    uint16_t maximum_rate_pps;
} cldt_stream_config_t;

typedef struct {
    cldt_deadline_queue_t *queue;
    cldt_stream_config_t streams[CLDT_ENDPOINT_MAX_STREAMS];
    size_t stream_count;
    cldt_policy_t active_policy;
    TaskHandle_t producer_task;
    TimerHandle_t release_timer;
    uint32_t random_state;
    bool running;
} cldt_workload_t;

esp_err_t cldt_workload_init(
    cldt_workload_t *workload,
    cldt_deadline_queue_t *queue,
    const cldt_stream_config_t *streams,
    size_t stream_count,
    uint32_t seed);

/* Starts release timers only after the run digest is accepted. */
esp_err_t cldt_workload_start(cldt_workload_t *workload, uint64_t run_start_local_us);

/* Applies a prevalidated immutable policy snapshot at a release boundary. */
esp_err_t cldt_workload_apply_policy(
    cldt_workload_t *workload,
    const cldt_policy_t *policy);

/* ISR entry: capture no payload and wake only the producer task. */
void cldt_workload_event_isr(void *context);

esp_err_t cldt_workload_stop(cldt_workload_t *workload);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/endpoint/main/workload.c`


In [ ]:
%%writefile /content/cldt_scratch/workload.c
#include "workload.h"

esp_err_t cldt_workload_init(
    cldt_workload_t *workload,
    cldt_deadline_queue_t *queue,
    const cldt_stream_config_t *streams,
    size_t stream_count,
    uint32_t seed)
{
    (void)workload;
    (void)queue;
    (void)streams;
    (void)stream_count;
    (void)seed;

    /*
     * IMPLEMENTATION TODO: validate non-null arguments, stream count, unique
     * stream IDs, payload/deadline/rate bounds, and aggregate offered rate against
     * the endpoint safety limit. Copy the approved stream list into workload-owned
     * storage, seed a documented deterministic jitter generator, and create the
     * producer task and timer with static allocation. A failed init must leave
     * running false and must not alter the deadline queue.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_workload_start(cldt_workload_t *workload, uint64_t run_start_local_us)
{
    (void)workload;
    (void)run_start_local_us;

    /*
     * IMPLEMENTATION TODO: require an accepted run start time and inactive
     * workload, calculate each first release from the same local monotonic epoch
     * plus its phase, and schedule absolute release intent rather than chaining
     * relative delays that accumulate jitter. Timer callbacks only notify the
     * producer task; payload creation, queue admission, tracing, and networking
     * happen in task context. Record release jitter against the intended time.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_workload_apply_policy(
    cldt_workload_t *workload,
    const cldt_policy_t *policy)
{
    (void)workload;
    (void)policy;

    /*
     * IMPLEMENTATION TODO: accept only a policy already authenticated and checked
     * by endpoint runtime, copy it into a staging snapshot, and swap it at one
     * documented release boundary so no stream sees half old/half new fields.
     * Revalidate that critical periods and reserved queue capacity remain inside
     * compiled limits. Trace old epoch, new epoch, and effective local time; do
     * not dynamically allocate or edit the manifest at runtime.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

void cldt_workload_event_isr(void *context)
{
    (void)context;

    /*
     * IMPLEMENTATION TODO: keep this ISR to the minimum allowed by FreeRTOS:
     * validate the stored context if practical, call the appropriate FromISR task
     * notification primitive, capture whether a higher-priority task woke, and
     * request a yield through the documented port macro. Do not allocate, log,
     * acquire a mutex, encode a frame, or call OpenThread from this ISR.
     */
}

esp_err_t cldt_workload_stop(cldt_workload_t *workload)
{
    (void)workload;

    /*
     * IMPLEMENTATION TODO: stop or disarm release timers, signal producer task
     * to stop creating new work, wait a bounded time for transport-owned slots,
     * explicitly expire remaining queued work if the deadline passes, and report
     * a final accounting snapshot. Only then set running false. Preserve the
     * reason for forced expiry so a fast shutdown never becomes invisible loss.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


`cldt_workload_apply_policy()` receives only a policy already authenticated, persisted, and locally validated. It still rechecks compiled critical/reserved limits.

1. One staging snapshot is copied and atomically swapped at a documented release boundary.
2. Exactly the predeclared bulk field changes. Critical, control, telemetry, queue reservation, and manifest source data remain unchanged.
3. Trace records old/new epoch, intended boundary, effective local time, and complete old/new policy digests.
4. TTL expiry uses the same safe-snapshot publication path; it is not a partial field rollback.
5. Duplicate epoch returns acknowledgement without a second apply.
6. Stop/fallback accounting continues to reconcile generated, queued, transport-owned, expired, and terminal work.


### Endpoint Transport and ACK Path
#### `firmware/endpoint/main/thread_transport.h`


In [ ]:
%%writefile /content/cldt_scratch/thread_transport.h
#ifndef CLDT_ENDPOINT_THREAD_TRANSPORT_H
#define CLDT_ENDPOINT_THREAD_TRANSPORT_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include "esp_err.h"

#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef void (*cldt_endpoint_command_fn)(
    void *context,
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint64_t received_local_us);

typedef struct {
    uint16_t local_port;
    uint16_t gateway_port;
    uint8_t gateway_ipv6[16];
    cldt_endpoint_command_fn on_command;
    void *callback_context;
} cldt_thread_transport_config_t;

esp_err_t cldt_thread_transport_init(
    const cldt_thread_transport_config_t *config);

/* Caller retains slot ownership until the completion result is returned. */
esp_err_t cldt_thread_transport_send(
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint32_t timeout_ms);

esp_err_t cldt_thread_transport_get_link(
    int8_t *rssi_dbm,
    uint8_t *link_quality,
    uint8_t *thread_role,
    uint32_t *partition_id);

esp_err_t cldt_thread_transport_stop(void);

#ifdef __cplusplus
}
#endif

#endif


#### `firmware/endpoint/main/thread_transport.c`


In [ ]:
%%writefile /content/cldt_scratch/thread_transport.c
#include "thread_transport.h"

esp_err_t cldt_thread_transport_init(
    const cldt_thread_transport_config_t *config)
{
    (void)config;

    /*
     * IMPLEMENTATION TODO: validate ports, gateway IPv6 address, callback, and
     * context; start the supported ESP-IDF OpenThread integration; commission or
     * attach using provisioned Thread credentials; wait for an attached state;
     * then bind one project UDP socket. Copy callback configuration into owned
     * state. On any failure, close the socket and unwind OpenThread in reverse
     * order; do not invent a parallel mesh implementation.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_transport_send(
    const uint8_t *datagram,
    size_t datagram_bytes,
    uint32_t timeout_ms)
{
    (void)datagram;
    (void)datagram_bytes;
    (void)timeout_ms;

    /*
     * IMPLEMENTATION TODO: reject null/oversized datagrams and zero or excessive
     * timeout values; transmit one whole UDP datagram; then return a precise send
     * outcome to the transport owner. UDP send success is not delivery success:
     * delivery acknowledgement and retry policy must be handled by the workload
     * state machine so counters distinguish sent, acknowledged, expired, and
     * dropped work. Never hold an OpenThread lock while waiting on a queue.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_transport_get_link(
    int8_t *rssi_dbm,
    uint8_t *link_quality,
    uint8_t *thread_role,
    uint32_t *partition_id)
{
    (void)rssi_dbm;
    (void)link_quality;
    (void)thread_role;
    (void)partition_id;

    /*
     * IMPLEMENTATION TODO: require all output pointers, acquire the OpenThread
     * API lock only long enough to read current RSSI, link quality, role, and
     * partition ID, copy scalar values, then release it before returning. A
     * detached or unavailable role is a valid observable state and should return
     * a clear status or sentinel, not stale data from a previous attachment.
     */
    return ESP_ERR_NOT_SUPPORTED;
}

esp_err_t cldt_thread_transport_stop(void)
{
    /*
     * IMPLEMENTATION TODO: reject new sends, unregister receive callbacks, close
     * the project UDP socket, then stop/deinitialize OpenThread as prescribed by
     * ESP-IDF. Clear internal callback state only after no callback can execute.
     * This order prevents a late OpenThread callback from dereferencing endpoint
     * runtime state that the supervisor has already reclaimed.
     */
    return ESP_ERR_NOT_SUPPORTED;
}


The receive callback only transfers bounded bytes/time into endpoint-owned work. Authentication, NVS commit, policy apply, and ACK serialization remain task context.

1. One command datagram is one complete UDP datagram; oversize/trailing bytes reject.
2. Callback data is copied before OpenThread releases it.
3. ACK/rejection is a separate project frame sent to the gateway with endpoint node/boot identity and command run/epoch.
4. UDP send success is not ACK success. Gateway waits a bounded time for A and B independently.
5. Transport stop unregisters callbacks before runtime state is reclaimed.


### Endpoint Entry Point
#### `firmware/endpoint/main/app_main.c`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_app_main.c
#include "esp_log.h"

static const char *TAG = "cldt_endpoint";

void app_main(void)
{
    ESP_LOGW(TAG,
             "Research scaffold only: endpoint runtime, deadline queue, "
             "workload, Thread transport, and power probe are not implemented.");

    /*
     * IMPLEMENTATION ORDER:
     * 1. Keep networking disabled while proving local software-timer, ISR,
     *    queue ownership, fixed-pool exhaustion, expiry, and counter tests.
     * 2. Add Thread attachment only after those tests produce reconciled traces.
     * 3. Add command handling only after protocol known-answer and replay tests.
     * 4. Enable the optional power probe last and document its overhead.
     * This entry point should remain small: construct runtime, call init/start,
     * and hand lifecycle ownership to the supervisor. It is not a demo script.
     */
}


The endpoint entry point follows the existing implementation order: local accounting and Thread path are prerequisites carried from earlier phases, while command handling opens only after protocol/auth/replay fixed vectors pass. It constructs one `cldt_endpoint_runtime_t`, calls init/start, and leaves command authentication, durable advancement, release-boundary apply, TTL expiry, and stop ownership inside the runtime. It does not contain a demo command, hard-coded key, fabricated epoch, or direct workload mutation.


### Endpoint Project Build Root
#### `firmware/endpoint/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_project_CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(EXTRA_COMPONENT_DIRS "${CMAKE_CURRENT_LIST_DIR}/../../common")

include($ENV{IDF_PATH}/tools/cmake/project.cmake)
project(cldt_endpoint)


This project root imports the same portable `common` component as host and gateway. Command authentication and replay support are added through the existing component/build owners, not by copying common source into the endpoint tree. Endpoint A and B may have distinct node/boot/provisioning state, but they compile the same command semantics from a clean build with archived IDF revision, full `sdkconfig`, partition table, and binary hash.


### Endpoint Build Boundary
#### `firmware/endpoint/main/CMakeLists.txt`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_CMakeLists.txt
idf_component_register(
    SRCS
        "app_main.c"
        "endpoint_runtime.c"
        "deadline_queue.c"
        "workload.c"
        "thread_transport.c"
        "power_probe.c"
    INCLUDE_DIRS "."
    REQUIRES
        common
        freertos
        nvs_flash
        esp_event
        esp_netif
        esp_timer
        driver
        openthread
)


#### `firmware/endpoint/main/Kconfig.projbuild`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_Kconfig.projbuild
menu "CLDT Endpoint"

config CLDT_ENDPOINT_NODE_ID
    int "Provisioning fallback node ID"
    range 1 4294967295
    default 100
    help
        Used only before the endpoint receives a provisioned identity. Record the
        final identity in run evidence; do not use a MAC address or an ad hoc hash.

choice CLDT_ENDPOINT_ROLE
    prompt "Endpoint role"
    default CLDT_ENDPOINT_ROUTER

config CLDT_ENDPOINT_ROUTER
    bool "Router-capable endpoint"
    help
        Enables the project role expected to be router-capable. Actual Thread role
        remains an observed run fact and must not be inferred from this selection.

config CLDT_ENDPOINT_LOW_POWER
    bool "Low-power end-device candidate"
    help
        Selects a candidate end-device configuration for a later admitted study.
        Actual Thread role remains observed, and no version-one energy claim is
        implied by this build choice.

endchoice

config CLDT_ENDPOINT_POOL_SLOTS
    int "Fixed message slots"
    range 8 128
    default 32
    help
        Compile-time queue-pool capacity. Choose from measured peak occupancy and
        preserve a control/critical reservation; never grow it only to hide loss.

config CLDT_ENDPOINT_MAX_TOTAL_RATE_PPS
    int "Compiled maximum aggregate application rate"
    range 1 500
    default 100
    help
        Hard local ceiling across all application streams. A ready manifest may
        request less, but no host policy can raise this value during a run.

config CLDT_ENDPOINT_EVENT_GPIO
    int "Local event input GPIO"
    range -1 30
    default -1
    help
        -1 disables the optional physical event input. Select a pin only after
        checking the exact board revision, boot strapping, pull mode, and debounce
        behavior; XIAO GPIO9 is a boot input and is not a safe generic default.

config CLDT_ENDPOINT_I2C_SDA_GPIO
    int "Optional power-probe I2C SDA GPIO"
    range 0 30
    default 22
    help
        XIAO ESP32-C6 D4/SDA is GPIO22. The version-one power probe is deferred;
        recheck the exact board and record wiring before a future energy pilot.

config CLDT_ENDPOINT_I2C_SCL_GPIO
    int "Optional power-probe I2C SCL GPIO"
    range 0 30
    default 23
    help
        XIAO ESP32-C6 D5/SCL is GPIO23. Do not assume another C6 board variant
        shares this mapping.

endmenu


#### `firmware/endpoint/sdkconfig.defaults`


In [ ]:
%%writefile /content/cldt_scratch/endpoint_sdkconfig.defaults
CONFIG_FREERTOS_HZ=1000
CONFIG_FREERTOS_USE_TRACE_FACILITY=y
CONFIG_FREERTOS_GENERATE_RUN_TIME_STATS=y
CONFIG_ESP_TASK_WDT_EN=y
CONFIG_OPENTHREAD_ENABLED=y
CONFIG_MBEDTLS_CHACHAPOLY_C=y
CONFIG_MBEDTLS_CHACHA20_C=y
CONFIG_MBEDTLS_POLY1305_C=y


The endpoint build already links NVS/OpenThread and enables ChaChaPoly. It still needs the actual authentication/replay owner represented in source and tests. The physical safety image keeps exact board role, pool size, compiled rate limit, channel, partition table, key-storage configuration, and binary digest. Endpoint A/B use matching command semantics; their node/boot/replay identities remain distinct.


## 12. Stale-Observation Manifest
### Authoring Companion: `experiments/authoring/stale-observation.jsonc`


In [ ]:
%%writefile /content/cldt_scratch/stale-observation.jsonc
{
  // This authoring copy plans a safety fallback test without disturbing RF or Thread routing.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stale-observation-fallback",
  "title": "Stale Observation Safety Fallback",
  "purpose": {
    "question": "Does the fidelity gate abstain and restore the local safe policy when host observations become stale?",
    "comparison": "Gated control before the observation pause versus the gateway state during and after the pause.",
    "primary_metric": "Time from stale-observation condition to recorded local fallback."
  },
  "setup": {
    // Reuse a fully validated physical baseline; identify the exact gateway-to-host publication path to pause.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Preserve the critical traffic profile while allowing one bounded bulk control surface.
    "streams": null
  },
  "scenario": {
    // Use observation_pause. Duration must exceed maximum_observation_age_ms with a recorded margin.
    "event": null,
    "at_s": null,
    "duration_s": null,
    // Name the publication gate or adapter that stops observations, not the Thread radio.
    "target": null
  },
  "treatment": {
    // Required after normal command path exists: gated_control, bulk_rate_reduce, host model and remote actuation enabled.
    "mode": null,
    "candidate_action": null,
    // Name a resolved profile with finite TTL and documented freshness/hysteresis limits.
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    "counter_reconciliation": null,
    // The same critical floor applies before and after fallback.
    "minimum_critical_on_time_pdr": null,
    // State the expected invalid condition, for example fallback not reached by the predeclared deadline.
    "negative_case": null
  },
  "evidence": {
    // Require newest observation, gate transition, command audit, fallback record, counters, and every recovery/requalification window.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Prove that only observations stop.",
      "method": "Add an application-level publication gate and dry-run it while physical endpoint traffic remains visible locally.",
      "done_when": "The pause boundary is traceable and no RF impairment is used."
    },
    {
      "path": "/scenario",
      "action": "Schedule stale data beyond the frozen freshness limit.",
      "method": "Measure normal observation age, record the profile limit, choose pause timing after warm-up, and use one injected fault.",
      "done_when": "Expected latest fallback time is recorded before the run."
    },
    {
      "path": "/treatment",
      "action": "Enable only a finite bounded bulk action.",
      "method": "Require gateway and endpoint acknowledgements for accept, reject, expiry, and fallback.",
      "done_when": "The evidence can reconstruct withdrawal, ABSTAIN-to-OBSERVE recovery, and either full requalification or continued abstention."
    }
  ]
}


### Strict Partner: `experiments/stale-observation.json`


In [ ]:
%%writefile /content/cldt_scratch/stale-observation.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stale-observation-fallback",
  "title": "Stale Observation Safety Fallback",
  "purpose": {
    "question": "Does the fidelity gate abstain and restore the local safe policy when host observations become stale?",
    "comparison": "Gated control before the observation pause versus the gateway state during and after the pause.",
    "primary_metric": "Time from stale-observation condition to recorded local fallback."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Reuse a validated stable Thread block and identify the precise gateway-to-host observation path that will be paused.",
      "method": "Freeze board roles, placement, channel, binaries, and normal observation cadence. Add an application-level publication gate at the gateway or host adapter; do not jam RF, stop endpoints, or modify Thread routing for this test.",
      "done_when": "A dry run shows physical endpoint traffic continues while the chosen observation stream stops at a traceable boundary."
    },
    {
      "path": "/scenario",
      "action": "Schedule a stale-observation interval that is longer than the calibrated freshness limit and short enough to observe recovery.",
      "method": "Measure normal observation age first, freeze the gate's freshness threshold in the versioned control profile, then choose pause time after warm-up and duration with margin beyond that threshold. Use only one fault in the run.",
      "done_when": "The ready manifest identifies observation_pause, its monotonic time, duration, and target; operator notes state the expected latest fallback time."
    },
    {
      "path": "/treatment",
      "action": "Enable only one finite, locally bounded action before proving its withdrawal.",
      "method": "Use bulk-rate reduction only, turn on the host model and remote actuation, set a finite command TTL in the control profile, keep a compiled safe static policy, and require gateway and endpoint acknowledgements for acceptance, rejection, expiry, and fallback.",
      "done_when": "The evidence includes last accepted observation, gate transition, command or expiry decision, local fallback, restored observations, the full requalification-window sequence, post-fallback counters, and a critical-service floor that the fallback must preserve."
    }
  ]
}


Both files remain template until normal command acceptance and fallback are already proven. Values come from archived baseline/profile/pilots, never from examples below.

| Existing manifest field | Source of actual value |
|---|---|
| `setup.nodes` | exact four admitted board labels/roles |
| `setup.thread_channel` | unchanged frozen physical block |
| `setup.placement` | unchanged placement/orientation/power description |
| `setup.firmware_reference` | short reference/digest to full `versions.json` |
| `execution.*` | non-reportable pilot durations and predeclared repetition plan |
| `traffic.streams` | validated stable workload with one unambiguous stream per actuated class |
| `scenario.event` | exactly `observation_pause` |
| `scenario.at_s` | after warm-up, inside measurement |
| `scenario.duration_s` | exceeds frozen observation-age limit with recovery margin |
| `scenario.target` | exact application publication boundary; not RF/Thread |
| `treatment.mode` | exactly `gated_control` |
| `treatment.candidate_action` | exactly `bulk_rate_reduce` |
| `treatment.control_profile` | resolved frozen safety profile |
| `treatment.host_model` / `remote_actuation` | true / true only when entry eligible |
| `acceptance.counter_reconciliation` | exactly true |
| `acceptance.minimum_critical_on_time_pdr` | unchanged frozen service floor |
| `acceptance.negative_case` | fallback/requalification failure rule selected before run |
| `evidence.*` | 4–8 bundle labels covering raw, gate, command, counters, versions, terminal evidence |

The trigger time, latest acceptable fallback time, restoration rule, requalification-window rule, and exact pause owner must be machine-bound. If schema/profile cannot carry them without prose ambiguity, strict JSON is not promoted.


## 13. Restart/Replay Manifest
### Authoring Companion: `experiments/authoring/restart-replay.jsonc`


In [ ]:
%%writefile /content/cldt_scratch/restart-replay.jsonc
{
  // This authoring copy tests command freshness through the normal gateway path.
  // It must never bypass authentication or call an endpoint apply function directly.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "restart-replay-safety",
  "title": "Restart And Command Replay Safety",
  "purpose": {
    "question": "Can a restarting endpoint reload durable replay state, reject stale or replayed policy epochs, and remain safe when that state is unavailable?",
    "comparison": "Command acceptance before restart, rejection after restart with valid durable state, and fail-safe behavior with missing or corrupt state.",
    "primary_metric": "Number of invalid policy applications after the deliberate restart."
  },
  "setup": {
    // Use a topology where one global authenticated finite-TTL command was durably recorded before apply and acknowledged.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    // Include stable operation, restart, reattachment, replay attempts, and fresh synchronization.
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Preserve the validated stable workload; do not add a load step to this safety case.
    "streams": null
  },
  "scenario": {
    // Use endpoint_restart and name the physical endpoint label that will be restarted.
    "event": null,
    "at_s": null,
    // Duration must cover reattachment and fresh synchronization, not merely power-cycle time.
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Safety mode may use only the already validated bounded bulk-rate action.
    "mode": null,
    "candidate_action": null,
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    // Reconciliation still applies; a safety test is not exempt from accounting.
    "counter_reconciliation": null,
    "minimum_critical_on_time_pdr": null,
    // Set the concrete negative outcome: any stale, replayed, or wrong-run policy application.
    "negative_case": null
  },
  "evidence": {
    // Require command audit with active run, coordinator plus old/new endpoint boot IDs, durable replay state, epoch, TTL, and per-attempt reason.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Establish normal global-command and persist-before-apply evidence before injecting a restart.",
      "method": "Commission a unique run and archive an accepted current command, replay-record commit, acknowledgement, endpoint boot ID, epoch, and TTL.",
      "done_when": "There is a pre-restart command and durable-state baseline against which rejection behavior can be interpreted."
    },
    {
      "path": "/scenario",
      "action": "Define one restart and a finite set of replay attempts.",
      "method": "Restart the selected endpoint, verify replay-record reload, then submit old-epoch, expired, wrong-run, wrong-authority-boot, corrupted-tag, and exact replay commands through the normal gateway path. Separately use a missing-or-corrupt-record fixture.",
      "done_when": "Operator notes list durable-state fixtures, exact invalid inputs, and expected rejection reasons."
    },
    {
      "path": "/acceptance",
      "action": "Predeclare zero invalid applications.",
      "method": "Require persist-before-apply, local fallback when replay state is unavailable, explicit new-run commissioning, and complete command/state evidence.",
      "done_when": "A reviewer can reconstruct every attempted command and durable-state transition."
    }
  ]
}


### Strict Partner: `experiments/restart-replay.json`


In [ ]:
%%writefile /content/cldt_scratch/restart-replay.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "restart-replay-safety",
  "title": "Restart And Command Replay Safety",
  "purpose": {
    "question": "Can a restarting endpoint reload durable replay state, reject stale or replayed policy epochs, and remain safe when that state is unavailable?",
    "comparison": "Command acceptance before restart, rejection after restart with valid durable state, and fail-safe behavior with missing or corrupt state.",
    "primary_metric": "Number of invalid policy applications after the deliberate restart."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Prove the normal global-command path, durable replay record, and boot identity observability before injecting a restart.",
      "method": "Reuse a valid safety-capable Thread topology, select the endpoint to restart, commission a unique run, and verify that one current authenticated finite-TTL global policy is durably recorded before it is applied and acknowledged.",
      "done_when": "A pre-restart trace contains active run identity, coordinator and endpoint boot IDs, accepted epoch, replay-record commit result, TTL, and acknowledgement reason."
    },
    {
      "path": "/scenario",
      "action": "Run one controlled restart and explicitly enumerate the replay attempts that follow it.",
      "method": "Choose endpoint_restart time after stable operation, capture duration through reattachment, verify the durable record reload, then send the ordinary gateway path an old epoch, an expired command, a wrong-run command, a wrong-authority-boot command, a corrupted tag, and an exact replay. Use a separate missing-or-corrupt-record fixture; never bypass authentication or call private apply functions directly.",
      "done_when": "The ready file fixes restart time and target, while operator notes list the replay-record fixtures, exact invalid command cases, and expected reject reasons."
    },
    {
      "path": "/acceptance",
      "action": "Make zero invalid policy applications the safety outcome and preserve complete command evidence.",
      "method": "Set safety treatment, select only the bounded bulk-rate action, require persist-before-apply, require the endpoint to keep its safe policy when replay state is unavailable, preserve critical service floor and counter reconciliation, and require command-audit, replay-state, raw-event, final-counter, version, and run-status artifacts.",
      "done_when": "A reviewer can reconstruct every attempted command and durable-state transition, confirm that stale, replayed, or wrong-run input changed no endpoint policy, and confirm that missing replay state required a new run."
    }
  ]
}


This is one endpoint-restart treatment, not a mixed restart campaign.

| Existing manifest field | Source of actual value |
|---|---|
| `setup.*` | same safety-capable four-board block and binary identities |
| `execution.*` | covers normal command, reset, attach, sync, replay cases, cooldown |
| `traffic.streams` | unchanged stable workload; no load step |
| `scenario.event` | exactly `endpoint_restart` |
| `scenario.at_s` | after one normal persist-before-apply acceptance |
| `scenario.duration_s` | reattachment and fresh clock-sync window |
| `scenario.target` | exact endpoint label |
| `treatment.mode` | exactly `safety` |
| `candidate_action` | exactly `bulk_rate_reduce`; no alternate action |
| `host_model` / `remote_actuation` | true / true only for eligible positive entry |
| `acceptance.negative_case` | any stale/replayed/wrong-run policy application |
| `evidence.required_artifacts` | command audit, replay record transitions, raw trace, counters, versions, terminal evidence |

The ready plan also machine-binds old epoch, expired, wrong run, wrong authority boot, corrupted tag, exact replay, and missing/corrupt record fixtures with expected status. Current schema has no structured list for those cases. They are not smuggled into an unrelated string; schema/contract repair precedes promotion.

Coordinator and gateway restart invariants also belong to Week-5 closure because DESIGN forbids resuming old actuated runs. No dedicated manifest exists for them. They remain deterministic/non-reportable restart fixtures with complete traces, not a fabricated ninth experiment or a second scenario inserted into this run.


## 14. Schema Boundary
### `schemas/experiment.schema.json`


In [ ]:
%%writefile /content/cldt_scratch/experiment.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "https://github.com/Reinathanajah/CLDT-Thread/blob/main/schemas/experiment.schema.json",
  "title": "CLDT Experiment Manifest",
  "description": "A deliberately small, two-stage contract. A template may retain null decisions and actionable _todo entries; a ready manifest may not.",
  "type": "object",
  "additionalProperties": false,
  "required": ["schema_version", "state", "experiment_id", "title", "purpose", "_todo"],
  "properties": {
    "schema_version": { "const": "2.0" },
    "state": { "enum": ["template", "ready"] },
    "experiment_id": { "$ref": "#/$defs/identifier" },
    "title": { "type": "string", "minLength": 8, "maxLength": 120 },
    "purpose": { "$ref": "#/$defs/purpose" },
    "_todo": {
      "type": "array",
      "uniqueItems": true,
      "items": { "$ref": "#/$defs/todo" }
    },
    "setup": { "$ref": "#/$defs/template_setup" },
    "execution": { "$ref": "#/$defs/template_execution" },
    "traffic": { "$ref": "#/$defs/template_traffic" },
    "scenario": { "$ref": "#/$defs/template_scenario" },
    "treatment": { "$ref": "#/$defs/template_treatment" },
    "acceptance": { "$ref": "#/$defs/template_acceptance" },
    "evidence": { "$ref": "#/$defs/template_evidence" }
  },
  "allOf": [
    {
      "if": { "properties": { "state": { "const": "template" } } },
      "then": { "properties": { "_todo": { "minItems": 1 } } }
    },
    {
      "if": { "properties": { "state": { "const": "ready" } } },
      "then": {
        "required": ["setup", "execution", "traffic", "scenario", "treatment", "acceptance", "evidence"],
        "properties": {
          "_todo": { "maxItems": 0 },
          "setup": { "$ref": "#/$defs/ready_setup" },
          "execution": { "$ref": "#/$defs/ready_execution" },
          "traffic": { "$ref": "#/$defs/ready_traffic" },
          "scenario": { "$ref": "#/$defs/ready_scenario" },
          "treatment": { "$ref": "#/$defs/ready_treatment" },
          "acceptance": { "$ref": "#/$defs/ready_acceptance" },
          "evidence": { "$ref": "#/$defs/ready_evidence" }
        }
      }
    }
  ],
  "$defs": {
    "identifier": {
      "type": "string",
      "minLength": 3,
      "maxLength": 64,
      "pattern": "^[a-z0-9][a-z0-9._-]*$"
    },
    "non_empty_text": {
      "type": "string",
      "minLength": 3,
      "maxLength": 512
    },
    "purpose": {
      "type": "object",
      "additionalProperties": false,
      "required": ["question", "comparison", "primary_metric"],
      "properties": {
        "question": { "$ref": "#/$defs/non_empty_text" },
        "comparison": { "$ref": "#/$defs/non_empty_text" },
        "primary_metric": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "todo": {
      "type": "object",
      "additionalProperties": false,
      "required": ["path", "action", "method", "done_when"],
      "properties": {
        "path": { "type": "string", "pattern": "^/" },
        "action": { "$ref": "#/$defs/non_empty_text" },
        "method": { "$ref": "#/$defs/non_empty_text" },
        "done_when": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "node": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "board", "role"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "board": { "enum": ["esp32-s3", "esp32-c6"] },
        "role": { "enum": ["gateway", "radio_coprocessor", "router_endpoint", "low_power_endpoint"] }
      }
    },
    "template_setup": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "nodes": { "type": ["array", "null"], "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/node" } },
        "thread_channel": { "type": ["integer", "null"], "minimum": 11, "maximum": 26 },
        "placement": { "type": ["string", "null"], "maxLength": 160 },
        "firmware_reference": { "type": ["string", "null"], "maxLength": 160 }
      }
    },
    "ready_setup": {
      "type": "object",
      "additionalProperties": false,
      "required": ["nodes", "thread_channel", "placement", "firmware_reference"],
      "properties": {
        "nodes": { "type": "array", "minItems": 4, "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/node" } },
        "thread_channel": { "type": "integer", "minimum": 11, "maximum": 26 },
        "placement": { "$ref": "#/$defs/non_empty_text" },
        "firmware_reference": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_execution": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "warmup_s": { "type": ["integer", "null"], "minimum": 5, "maximum": 300 },
        "measurement_s": { "type": ["integer", "null"], "minimum": 30, "maximum": 3600 },
        "cooldown_s": { "type": ["integer", "null"], "minimum": 5, "maximum": 300 },
        "repetitions": { "type": ["integer", "null"], "minimum": 1, "maximum": 30 },
        "seed": { "type": ["integer", "null"], "minimum": 0, "maximum": 4294967295 }
      }
    },
    "ready_execution": {
      "type": "object",
      "additionalProperties": false,
      "required": ["warmup_s", "measurement_s", "cooldown_s", "repetitions", "seed"],
      "properties": {
        "warmup_s": { "type": "integer", "minimum": 5, "maximum": 300 },
        "measurement_s": { "type": "integer", "minimum": 30, "maximum": 3600 },
        "cooldown_s": { "type": "integer", "minimum": 5, "maximum": 300 },
        "repetitions": { "type": "integer", "minimum": 1, "maximum": 30 },
        "seed": { "type": "integer", "minimum": 0, "maximum": 4294967295 }
      }
    },
    "stream": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "source", "class", "period_ms", "payload_bytes", "deadline_ms", "burst_packets"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "source": { "$ref": "#/$defs/identifier" },
        "class": { "enum": ["control", "critical", "telemetry", "bulk"] },
        "period_ms": { "type": ["integer", "null"], "minimum": 10, "maximum": 3600000 },
        "payload_bytes": { "type": ["integer", "null"], "minimum": 1, "maximum": 256 },
        "deadline_ms": { "type": ["integer", "null"], "minimum": 10, "maximum": 3600000 },
        "burst_packets": { "type": ["integer", "null"], "minimum": 1, "maximum": 100 }
      }
    },
    "ready_stream": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "source", "class", "period_ms", "payload_bytes", "deadline_ms", "burst_packets"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "source": { "$ref": "#/$defs/identifier" },
        "class": { "enum": ["control", "critical", "telemetry", "bulk"] },
        "period_ms": { "type": "integer", "minimum": 10, "maximum": 3600000 },
        "payload_bytes": { "type": "integer", "minimum": 1, "maximum": 256 },
        "deadline_ms": { "type": "integer", "minimum": 10, "maximum": 3600000 },
        "burst_packets": { "type": "integer", "minimum": 1, "maximum": 100 }
      }
    },
    "template_traffic": {
      "type": "object",
      "additionalProperties": false,
      "properties": { "streams": { "type": ["array", "null"], "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/stream" } } }
    },
    "ready_traffic": {
      "type": "object",
      "additionalProperties": false,
      "required": ["streams"],
      "properties": { "streams": { "type": "array", "minItems": 1, "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/ready_stream" } } }
    },
    "template_scenario": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "event": { "enum": ["none", "load_step", "observation_pause", "endpoint_restart", "topology_shift", null] },
        "at_s": { "type": ["integer", "null"], "minimum": 0, "maximum": 3600 },
        "duration_s": { "type": ["integer", "null"], "minimum": 0, "maximum": 3600 },
        "target": { "type": ["string", "null"], "maxLength": 120 }
      }
    },
    "ready_scenario": {
      "type": "object",
      "additionalProperties": false,
      "required": ["event", "at_s", "duration_s", "target"],
      "properties": {
        "event": { "enum": ["none", "load_step", "observation_pause", "endpoint_restart", "topology_shift"] },
        "at_s": { "type": "integer", "minimum": 0, "maximum": 3600 },
        "duration_s": { "type": "integer", "minimum": 0, "maximum": 3600 },
        "target": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_treatment": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "mode": { "enum": ["baseline", "prediction", "gated_control", "safety", "smp", "power", null] },
        "candidate_action": { "enum": ["none", "bulk_rate_reduce", "phase_stagger", "power_profile", null] },
        "control_profile": { "type": ["string", "null"], "maxLength": 160 },
        "host_model": { "type": ["boolean", "null"] },
        "remote_actuation": { "type": ["boolean", "null"] }
      }
    },
    "ready_treatment": {
      "type": "object",
      "additionalProperties": false,
      "required": ["mode", "candidate_action", "control_profile", "host_model", "remote_actuation"],
      "properties": {
        "mode": { "enum": ["baseline", "prediction", "gated_control", "safety", "smp", "power"] },
        "candidate_action": { "enum": ["none", "bulk_rate_reduce", "phase_stagger", "power_profile"] },
        "control_profile": { "$ref": "#/$defs/non_empty_text" },
        "host_model": { "type": "boolean" },
        "remote_actuation": { "type": "boolean" }
      }
    },
    "template_acceptance": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "counter_reconciliation": { "type": ["boolean", "null"] },
        "minimum_critical_on_time_pdr": { "type": ["number", "null"], "minimum": 0, "maximum": 1 },
        "negative_case": { "type": ["string", "null"], "maxLength": 120 }
      }
    },
    "ready_acceptance": {
      "type": "object",
      "additionalProperties": false,
      "required": ["counter_reconciliation", "minimum_critical_on_time_pdr", "negative_case"],
      "properties": {
        "counter_reconciliation": { "const": true },
        "minimum_critical_on_time_pdr": { "type": "number", "minimum": 0, "maximum": 1 },
        "negative_case": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_evidence": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "required_artifacts": { "type": ["array", "null"], "maxItems": 8, "uniqueItems": true, "items": { "$ref": "#/$defs/non_empty_text" } },
        "operator_notes_required": { "type": ["boolean", "null"] },
        "topology_photo_required": { "type": ["boolean", "null"] }
      }
    },
    "ready_evidence": {
      "type": "object",
      "additionalProperties": false,
      "required": ["required_artifacts", "operator_notes_required", "topology_photo_required"],
      "properties": {
        "required_artifacts": { "type": "array", "minItems": 4, "maxItems": 8, "uniqueItems": true, "items": { "$ref": "#/$defs/non_empty_text" } },
        "operator_notes_required": { "const": true },
        "topology_photo_required": { "type": "boolean" }
      }
    }
  }
}


Schema work remains narrow and safety-specific:

1. Ready node ID and role uniqueness is enforced by cross-field admission because `uniqueItems` alone rejects only identical objects.
2. Safe treatment combinations are enforced: stale/gated-control/observation-pause and safety/endpoint-restart.
3. Actuated streams have a deterministic class mapping.
4. Safety plans retain structured trigger/fallback/requalification/replay-case expectations after `_todo` becomes empty.
5. Command-key identity is non-secret evidence; secret material is rejected.
6. Runtime string limits stay no larger than fixed C buffers.
7. Unknown fields remain rejected. A schema extension is versioned and mirrored in both authoring/strict partners and parser tests.
8. Syntax/schema PASS still does not bypass positive Week-4 evidence, profile digest, binary identity, device provisioning, or deterministic safety tests.


## 15. CI and Git Closure
### `.github/workflows/scaffold-validation.yml`


In [ ]:
%%writefile /content/cldt_scratch/scaffold-validation.yml
name: Scaffold Validation

on:
  push:
    branches:
      - main
  pull_request:

permissions:
  contents: read

jobs:
  host-and-manifests:
    name: Host Build and Manifest Contracts
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Configure host scaffold
        run: cmake -S . -B build -DCLDT_BUILD_TESTS=ON

      - name: Build host scaffold
        run: cmake --build build --parallel

      - name: Run test harness
        run: ctest --test-dir build --output-on-failure

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install JSON Schema validator
        run: python -m pip install --disable-pip-version-check "jsonschema==4.26.0"

      - name: Validate experiment manifests
        shell: bash
        run: |
          python - <<'PY'
          import json
          from pathlib import Path

          from jsonschema import Draft202012Validator

          schema_path = Path("schemas/experiment.schema.json")
          schema = json.loads(schema_path.read_text(encoding="utf-8"))
          Draft202012Validator.check_schema(schema)
          validator = Draft202012Validator(schema)

          strict_files = sorted(Path("experiments").glob("*.json"))
          if not strict_files:
              raise SystemExit("No strict experiment manifests were found.")

          strict_ids = {}
          for path in strict_files:
              document = json.loads(path.read_text(encoding="utf-8"))
              errors = sorted(validator.iter_errors(document), key=lambda error: list(error.path))
              if errors:
                  for error in errors:
                      location = "/" + "/".join(str(part) for part in error.path)
                      print(f"{path}:{location}: {error.message}")
                  raise SystemExit(f"Schema validation failed for {path}.")
              strict_ids[path.stem] = document["experiment_id"]

          authoring_files = sorted(Path("experiments/authoring").glob("*.jsonc"))
          if len(authoring_files) != len(strict_files):
              raise SystemExit("Strict JSON and JSONC authoring manifest counts differ.")

          for path in authoring_files:
              uncommented = "\n".join(
                  line for line in path.read_text(encoding="utf-8").splitlines()
                  if not line.lstrip().startswith("//")
              )
              document = json.loads(uncommented)
              if document["experiment_id"] != strict_ids.get(path.stem):
                  raise SystemExit(f"Experiment ID mismatch for {path}.")

          print(f"Validated {len(strict_files)} strict manifests and {len(authoring_files)} authoring copies.")
          PY


The workflow currently proves only scaffold build, skipped tests, and manifest shape. Week-5 closure updates CI so active safety tests actually execute and missing crypto dependencies fail. Firmware hardware behavior cannot be replaced by CI, but host fixed vectors, portable guard/policy logic, schema pairs, secret scanning, and notebook JSON validity can be deterministic.

Commit discipline remains separated:

1. Contract/header/schema changes.
2. Portable implementation and tests.
3. Host integration.
4. Gateway firmware.
5. Endpoint firmware.
6. Ready manifest promotion.
7. Physical evidence remains outside Git where ignored/private, with public derived artifacts only when safe.

No secret, `sdkconfig` containing credentials, private IP, raw key, or unredacted broker configuration enters a commit.


## 16. Hardware Preparation
### Same Four Boards, No New Purchase

| Physical owner | Week-5 action | Closure evidence |
|---|---|---|
| S3 gateway | flash exact safety image; verify RCP, Thread, Wi-Fi, broker, guard disarmed at boot | binary/sdkconfig hashes, state trace |
| C6 RCP | retain upstream image and UART wiring | unchanged hash, attach/cold-boot record |
| Endpoint A/B | flash exact endpoint safety image; commission identity/key/replay state | binary hashes, boot IDs, replay-load result |
| Powered hub/cables | simultaneous activity soak without reconnect/brownout | serial continuity and power notes |
| Private AP/host | local broker path and monotonic recorder | interface/topic contract without secret values |
| Optional logic analyzer | observe UART/GPIO correlation only if already stable | capture metadata; never RF ground truth |

Physical sequence:

1. Command-key provisioning occurs off-run with physical presence. Host uses an ignored secret source; gateway and endpoints use the documented local mechanism. Only key identity is recorded.
2. BLE/HTTP provisioning is stopped before warm-up.
3. `CONFIG_CLDT_GATEWAY_REMOTE_ACTUATION` remains off for cold boot and rejection dry-runs.
4. The exact safety image with the switch enabled is flashed only after host/S3/A/B fixed vectors and failure cases pass.
5. One no-action command-path dry run proves topic, raw bytes, gateway rejection, endpoint rejection, ACK, and recorder joins.
6. One non-reportable valid action pilot proves both endpoints persist then apply the same epoch at a release boundary.
7. Physical placement, channel, RCP wiring, endpoint roles/parents/partition, and power path stay unchanged.


## 17. Normal One-Action Pilot
### Required Before Fault Injection

One ordinary command must succeed before stale or replay failure has an interpretable baseline.

1. Warm-up and prior completed horizons make the cross-layer gate pass through the full `COLD -> OBSERVE -> TRUSTED` sequence.
2. Current policy and exact reduced bulk policy are archived before issuance.
3. Host reserves epoch, gateway-time issue value, and finite TTL; one immutable command frame is authenticated and recorded.
4. Gateway rejects retained/duplicate/invalid input, then accepts the valid raw datagram under local health and limits.
5. The identical bytes reach A and B.
6. Each endpoint authenticates, checks authority/run/epoch/TTL/clock/local limits, commits and reads back replay state, then applies at the declared release boundary.
7. Two endpoint ACKs and the gateway decision join the host command audit.
8. Critical-service floor, item audit, and aggregate reconciliation still pass.

| Pilot record | Example format |
|---|---|
| Command digest / epoch / TTL | e.g. digest, integer, milliseconds |
| Gateway decision | e.g. accepted with exact reason |
| Endpoint A persistence/apply/ACK | e.g. three timestamps + status |
| Endpoint B persistence/apply/ACK | e.g. three timestamps + status |
| Bulk field before/after | e.g. exact integers |
| Critical fields | e.g. byte-identical |
| Critical service result | e.g. ratio vs frozen floor |
| Reconciliation | e.g. item and aggregate PASS |

Without this baseline, fault injection stops. A fallback trace cannot compensate for a command path that never worked normally.


## 18. Stale Observation, Fallback, dan Requalification

The pause is application-level and affects only the manifest target.

1. Normal gated control and both endpoint ACKs occur first.
2. At the frozen host-monotonic boundary, the named gateway/host publication path stops delivering observations while Thread traffic, endpoint work, and topology continue.
3. The newest accepted observation timestamp remains unchanged and age becomes stale through checked arithmetic.
4. Gate enters `ABSTAIN` on the first failing evaluation and stops new proposals.
5. The chosen local mechanism is recorded honestly: gateway-local fallback when the gateway itself detects the condition, or finite endpoint TTL expiry when only the host loses observations. These latencies are not conflated.
6. Both endpoints reach compiled safe policy no later than the predeclared bound and acknowledge/trace the transition.
7. Observation publication resumes at the frozen time. First clean sample enters `OBSERVE`; only the full passing-window sequence can restore `TRUSTED`.
8. Any failure during requalification returns to `ABSTAIN` and resets the documented sequence.
9. A new command, if eligible, uses a strictly higher epoch. Old pending bytes never resume.
10. Raw trace, command audit, gate trace, counters, and terminal status reconcile.

Primary latency is reported with trigger/fallback clock relationship and uncertainty. A gate that abstains correctly but never becomes useful reports trusted-horizon fraction beside safety.


## 19. Endpoint Restart and Replay Rejection

One selected endpoint resets; gateway, RCP, other endpoint, channel, placement, workload, and active run stay fixed for the valid-record case.

1. A normal command is persisted/applied/acknowledged before reset.
2. Reset occurs at the declared boundary without unplugging or moving other boards.
3. New endpoint `boot_id` appears; the same valid replay record reloads and is checked before remote apply.
4. Old epoch, expired command, wrong run, wrong authority boot, corrupted tag, and exact byte replay travel through the normal gateway/Thread path.
5. Every attempt records raw bytes/digest, expected reason, actual gateway reason, actual endpoint reason, replay record before/after, and policy before/after.
6. No rejected case changes active policy or highest applied epoch.
7. A fresh strictly higher command is accepted only after attachment, valid clock sync, and normal authorization.
8. Missing/corrupt replay state is a separate non-reportable fixture. Only the project replay key/fixture is affected; secret/provisioning namespaces and prior evidence are preserved.
9. Missing/corrupt/unwritable state keeps safe policy and requires explicit new-run commissioning.
10. Zero invalid policy applications is the acceptance criterion.

Host and gateway restart fixtures separately prove that old actuated run forwarding is not resumed. They allocate a new run and coordinator identity; loss of ledger continuity also rotates the key before later actuation.


## 20. Failure and Cut Rules

| Failure | Immediate classification | What remains allowed |
|---|---|---|
| Week-4 outcome not positive/evaluable | physical actuation ineligible | deterministic gate/auth/guard/replay work |
| Any active test skipped/not implemented | safety path incomplete | source/test repair within Week 5 |
| Auth vector mismatch | no command key provisioned | fixed-vector debugging only |
| Profile/action/TTL ambiguous | manifest remains template | contract/schema repair |
| One endpoint missing ACK | command not globally successful | preserve partial raw evidence |
| Persist/readback fails | endpoint rejects; safe policy | replay failure evidence |
| Stale path misses fallback deadline | safety run invalid/negative | remote disabled; evidence retained |
| Requalification shortcuts hysteresis | invariant failed | remote disabled |
| Replay case changes policy | safety failure | remote disabled; no Week-6 feature rescue |
| Counter/item audit mismatch | performance interpretation invalid | raw diagnostics only |
| Unexpected reset/topology/power event | invalid/interrupted | preserve run and reason |
| Work unfinished on 19 October | no feature spill into Week 6 | limitation, reproduction, presentation |

A negative, complete safety result is reportable. An incomplete safety path is not converted into a closed-loop claim.


## Phase-5 Delivery Checklist
### Closure Required Before Week 6

- [ ] Phase-4 positive/evaluable entry is archived, or physical actuation is explicitly disabled.
- [ ] Immutable model artifact and runtime `model_revision` have non-conflicting gate semantics.
- [ ] Gate sample/limits represent issuance ordering, covariance input/limit, integrity, freshness, coverage, clock, and calibrated region.
- [ ] Authenticated plaintext contract uses 132-byte AAD, zero plaintext, 12-byte run/epoch nonce, and fixed vectors on host/S3/A/B.
- [ ] Native and ESP-IDF crypto dependencies are pinned and linked.
- [ ] Policy payload has one portable encode/decode owner and fixed bytes.
- [ ] Exact rejection statuses remain distinguishable through wire, recorder, and reproduction.
- [ ] Host broker publishes non-retained immutable command bytes and receives bounded gateway/endpoint decisions under the frozen topic/QoS contract.
- [ ] Exactly one bulk field/value and one finite TTL are frozen in the resolved profile/selection.
- [ ] Valid no-proposal outcome is explicit.
- [ ] Gate state/reason/hysteresis tests and guard rejection tests execute without skip.
- [ ] Gateway retains immutable raw command bytes and rejects retained/reassembled-invalid MQTT input.
- [ ] Gateway and endpoint `app_main.c` start one supervisor-owned runtime; no scaffold-only return or parallel demo path remains.
- [ ] Gateway forwards identical command bytes to two admitted endpoint addresses.
- [ ] Endpoint command-key and commissioning owners exist.
- [ ] Replay record is versioned, integrity-checked, committed/read back before apply, and fail-closed on missing/corrupt/unwritable state.
- [ ] Endpoint policy expiry has a supervisor owner and returns to compiled safe policy by TTL.
- [ ] Both safety manifest pairs contain only real values, have empty `_todo`, and pass schema plus cross-field/runtime-buffer admission.
- [ ] One normal command pilot has gateway plus two endpoint acknowledgements and preserves critical service.
- [ ] Stale observation produces ABSTAIN, local safe policy, complete recovery trace, and full requalification sequence.
- [ ] Restart/replay cases produce exact rejection reasons and zero invalid applications.
- [ ] Coordinator/gateway restart fixtures never resume the old actuated run.
- [ ] Raw command/gate/replay events pass item audit, aggregate reconciliation, and Week-5 reproduction.
- [ ] Every complete/invalid/interrupted run remains in the ledger.
- [ ] Topology shift, ablation, SMP, power, dashboard, extra node, new action, and new purchase remain untouched.
- [ ] Feature freeze occurs by 19 October; Week 6 begins 20 October with evidence repair only.

> **Exit rule:** Week 5 closes only with complete fail-closed evidence. Any unproven authentication, authority, epoch, TTL, persistence, rejection, fallback, or requalification invariant keeps remote actuation disabled throughout Week 6.
